<div style="background-color: black;">
    <hr style="border: 4px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue;">
    AVAILABILITY FACTORS DATA PROCESSING
    <br>
    SOLAR | WIND OFFSHORE | WIND ONSHORE | RUN OF RIVER
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<br>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
This script is used to process the raw time series data for renewable source availability factors obtained from the PyPSA model simulations conducted for the Dispa-SET Unleashed project.
    <br>
The explanation text cells detail the entire process and help the reader understand how the final results were achieved step-by-step
</div>
<br>
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [1]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple
import re
from IPython.display import HTML
from difflib import get_close_matches

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [2]:
"""
# List of countries with their acronyms in parentheses

countries = [

    "Albania       (AL)"  , "Armenia        (AM)"  , "Austria          (AT)"  , "Azerbaijan (AZ)"  ,
    "Belarus       (BY)"  , "Belgium        (BE)"  , "Bosnia and Herz. (BA)"  , "Bulgaria   (BG)"  ,
    "Croatia       (HR)"  , "Cyprus         (CY)"  , "Czech Republic   (CZ)"  , "Denmark    (DK)"  ,
    "Estonia       (EE)"  , "Finland        (FI)"  , "France           (FR)"  , "Georgia    (GE)"  ,
    "Germany       (DE)"  , "Greece         (EL)"  , "Hungary          (HU)"  , "Iceland    (IS)"  ,
    "Ireland       (IE)"  , "Italy          (IT)"  , "Kosovo           (XK)"  , "Latvia     (LV)"  ,
    "Lithuania     (LT)"  , "Luxembourg     (LU)"  , "Malta            (MT)"  , "Moldova    (MD)"  ,
    "Montenegro    (ME)"  , "Netherlands    (NL)"  , "North Macedonia  (MK)"  , "Norway     (NO)"  , 
    "Poland        (PL)"  , "Portugal       (PT)"  , "Romania          (RO)"  , "Russia     (RU)"  , 
    "Russia Legacy (RU)"  , "Serbia         (RS)"  , "Slovakia         (SK)"  , "Slovenia   (SI)"  , 
    "Spain         (ES)"  , "Sweden         (SE)"  , "Switzerland      (CH)"  , "Turkey     (TR)"  , 
    "Ukraine       (UA)"  , "United Kingdom (UK)"

]

# Set the path where you want to create the folders
# Replace 'C:/Your/Path/Here' with your desired directory
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/PowerPlants'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'(.∗?)(.*?)', country_string)
    
    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1)
        folder_path = os.path.join(base_path, acronym)
        
        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")

print("\nAll folders created successfully!")
"""

'\n# List of countries with their acronyms in parentheses\n\ncountries = [\n\n    "Albania       (AL)"  , "Armenia        (AM)"  , "Austria          (AT)"  , "Azerbaijan (AZ)"  ,\n    "Belarus       (BY)"  , "Belgium        (BE)"  , "Bosnia and Herz. (BA)"  , "Bulgaria   (BG)"  ,\n    "Croatia       (HR)"  , "Cyprus         (CY)"  , "Czech Republic   (CZ)"  , "Denmark    (DK)"  ,\n    "Estonia       (EE)"  , "Finland        (FI)"  , "France           (FR)"  , "Georgia    (GE)"  ,\n    "Germany       (DE)"  , "Greece         (EL)"  , "Hungary          (HU)"  , "Iceland    (IS)"  ,\n    "Ireland       (IE)"  , "Italy          (IT)"  , "Kosovo           (XK)"  , "Latvia     (LV)"  ,\n    "Lithuania     (LT)"  , "Luxembourg     (LU)"  , "Malta            (MT)"  , "Moldova    (MD)"  ,\n    "Montenegro    (ME)"  , "Netherlands    (NL)"  , "North Macedonia  (MK)"  , "Norway     (NO)"  , \n    "Poland        (PL)"  , "Portugal       (PT)"  , "Romania          (RO)"  , "Russia     (RU)"  , \n  

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [3]:
# Get the current working directory
current_directory = os.getcwd()

# Navigate to the parent directory of "Dispa-SET_Unleash"
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# Get the path to the "Dispa-SET_Unleash" folder
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# Construct the dispaSET_unleash_folder_name variable
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.1. PyPSA Source Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
There are two sccenarios as source of PyPSA power plants raw data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
Reference_Scenario
<li>
Suficiency_Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The scenario variable must be selected before proceeding to the next processing steps
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [4]:
pypsa_scenario = "Reference_Scenario"
#pypsa_scenario = "Suficiency_Scenario"

print("PyPSA Chosen Scenario:", pypsa_scenario)

PyPSA Chosen Scenario: Reference_Scenario


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.2. Secondary Folders Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The many subfolders within the Unleash directory require their location paths to be defined for correct access during processing.
<br>
All of these are dependent on the chosen scenario.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [5]:
# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the data used as base
additional_path_1 = "/Database/AvailabilityFactors"
availability_factors_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# Construct the Dispa-SET_Unleash_Availability_Factor_name variable of the data used as reference
availability_factors_base_data_folder_name = os.path.basename(availability_factors_base_data_folder_path)

print("availability_factors_base_data_folder_name:", availability_factors_base_data_folder_name)
print("availability_factors_base_data_folder_path:", availability_factors_base_data_folder_path)
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the PyPSA raw data
additional_path_2 = os.path.join("RawData_PyPSA", pypsa_scenario, "dispatch")

availability_factors_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_2

# Construct the Dispa-SET_Unleash_Availability_Factor_folder_name variable of the PyPSA raw data
availability_factors_pypsa_raw_data_folder_name = os.path.basename(availability_factors_pypsa_raw_data_folder_path)

print("availability_factors_pypsa_raw_data_folder_name:", availability_factors_pypsa_raw_data_folder_name)
print("availability_factors_pypsa_raw_data_folder_path:", availability_factors_pypsa_raw_data_folder_path)
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the PyPSA formated data
additional_path_3 = os.path.join("Database_PyPSA", pypsa_scenario, "AvailabilityFactors")

availability_factors_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_3

# Construct the Dispa-SET_Unleash_Availability_Factor_folder_name variable of the PyPSA formated data
availability_factors_pypsa_formated_data_folder_name = os.path.basename(availability_factors_pypsa_formated_data_folder_path)

print("availability_factors_pypsa_formated_data_folder_name:", availability_factors_pypsa_formated_data_folder_name)
print("availability_factors_pypsa_formated_data_folder_path:", availability_factors_pypsa_formated_data_folder_path)

availability_factors_base_data_folder_name: AvailabilityFactors
availability_factors_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors
availability_factors_pypsa_raw_data_folder_name: dispatch
availability_factors_pypsa_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/dispatch
availability_factors_pypsa_formated_data_folder_name: AvailabilityFactors
availability_factors_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/AvailabilityFactors


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [6]:
# List of folder names to be addressed
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

print("zone_names:", zone_names)

zone_names: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary specifies the possible alternative names—or synonyms/aliases—used in international nomenclature for the EU countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [15]:
zone_names_equivalences_dict = {

"AL"  :  {"Acronym": ["  "       ] ,   "name": ["Albania   "       ]},  
"AM"  :  {"Acronym": ["  "       ] ,   "name": ["Armenia   "       ]},
"AT"  :  {"Acronym": ["  "       ] ,   "name": ["Austria   "       ]},
"AZ"  :  {"Acronym": ["  "       ] ,   "name": ["Azerbaijan"       ]},
"BY"  :  {"Acronym": ["  "       ] ,   "name": ["Belarus"          ]},
"BE"  :  {"Acronym": ["  "       ] ,   "name": ["Belgium"          ]},
"BA"  :  {"Acronym": ["  "       ] ,   "name": ["Bosnia and Herz." ]},
"BG"  :  {"Acronym": ["  "       ] ,   "name": ["Bulgaria"         ]},
"HR"  :  {"Acronym": ["  "       ] ,   "name": ["Croatia"          ]},
"CY"  :  {"Acronym": ["  "       ] ,   "name": ["Cyprus"           ]},
"CZ"  :  {"Acronym": ["  "       ] ,   "name": ["Czech Republic"   ]},
"DK"  :  {"Acronym": ["  "       ] ,   "name": ["Denmark"          ]},
"EE"  :  {"Acronym": ["  "       ] ,   "name": ["Estonia"          ]},
"FI"  :  {"Acronym": ["  "       ] ,   "name": ["Finland"          ]},
"FR"  :  {"Acronym": ["  "       ] ,   "name": ["France"           ]},
"GE"  :  {"Acronym": ["  "       ] ,   "name": ["Georgia"          ]}, 
"DE"  :  {"Acronym": ["  "       ] ,   "name": ["Germany"          ]},
"EL"  :  {"Acronym": ["GR"       ] ,   "name": ["Greece"           ]},
"HU"  :  {"Acronym": ["  "       ] ,   "name": ["Hungary"          ]},
"IS"  :  {"Acronym": ["  "       ] ,   "name": ["Iceland"          ]},
"IE"  :  {"Acronym": ["  "       ] ,   "name": ["Ireland"          ]},
"IT"  :  {"Acronym": ["  "       ] ,   "name": ["Italy"            ]},
"XK"  :  {"Acronym": ["  "       ] ,   "name": ["Kosovo"           ]},
"LV"  :  {"Acronym": ["  "       ] ,   "name": ["Latvia"           ]},
"LT"  :  {"Acronym": ["  "       ] ,   "name": ["Lithuania"        ]},
"LU"  :  {"Acronym": ["  "       ] ,   "name": ["Luxembourg"       ]},
"MT"  :  {"Acronym": ["  "       ] ,   "name": ["Malta"            ]},
"MD"  :  {"Acronym": ["  "       ] ,   "name": ["Moldova"          ]},
"ME"  :  {"Acronym": ["  "       ] ,   "name": ["Montenegro"       ]},
"NL"  :  {"Acronym": ["  "       ] ,   "name": ["Netherlands"      ]},
"MK"  :  {"Acronym": ["  "       ] ,   "name": ["North Macedonia"  ]},
"NO"  :  {"Acronym": ["  "       ] ,   "name": ["Norway"           ]},
"PL"  :  {"Acronym": ["  "       ] ,   "name": ["Poland"           ]},
"PT"  :  {"Acronym": ["  "       ] ,   "name": ["Portugal"         ]},
"RO"  :  {"Acronym": ["  "       ] ,   "name": ["Romania"          ]},
"RU"  :  {"Acronym": ["  "       ] ,   "name": ["Russia"           ]},
"RS"  :  {"Acronym": ["  "       ] ,   "name": ["Serbia"           ]},
"SK"  :  {"Acronym": ["  "       ] ,   "name": ["Slovakia"         ]},
"SI"  :  {"Acronym": ["  "       ] ,   "name": ["Slovenia"         ]},
"ES"  :  {"Acronym": ["  "       ] ,   "name": ["Spain"            ]},
"SE"  :  {"Acronym": ["  "       ] ,   "name": ["Sweden"           ]},
"CH"  :  {"Acronym": ["  "       ] ,   "name": ["Switzerland"      ]},
"TR"  :  {"Acronym": ["  "       ] ,   "name": ["Turkey"           ]},
"UA"  :  {"Acronym": ["  "       ] ,   "name": ["Ukraine"          ]},
"UK"  :  {"Acronym": ["GB"       ] ,   "name": ["United Kingdom"   ]},
       
}

zone_names_equivalences_dict

{'AL': {'Acronym': ['  '], 'name': ['Albania   ']},
 'AM': {'Acronym': ['  '], 'name': ['Armenia   ']},
 'AT': {'Acronym': ['  '], 'name': ['Austria   ']},
 'AZ': {'Acronym': ['  '], 'name': ['Azerbaijan']},
 'BY': {'Acronym': ['  '], 'name': ['Belarus']},
 'BE': {'Acronym': ['  '], 'name': ['Belgium']},
 'BA': {'Acronym': ['  '], 'name': ['Bosnia and Herz.']},
 'BG': {'Acronym': ['  '], 'name': ['Bulgaria']},
 'HR': {'Acronym': ['  '], 'name': ['Croatia']},
 'CY': {'Acronym': ['  '], 'name': ['Cyprus']},
 'CZ': {'Acronym': ['  '], 'name': ['Czech Republic']},
 'DK': {'Acronym': ['  '], 'name': ['Denmark']},
 'EE': {'Acronym': ['  '], 'name': ['Estonia']},
 'FI': {'Acronym': ['  '], 'name': ['Finland']},
 'FR': {'Acronym': ['  '], 'name': ['France']},
 'GE': {'Acronym': ['  '], 'name': ['Georgia']},
 'DE': {'Acronym': ['  '], 'name': ['Germany']},
 'EL': {'Acronym': ['GR'], 'name': ['Greece']},
 'HU': {'Acronym': ['  '], 'name': ['Hungary']},
 'IS': {'Acronym': ['  '], 'name': ['Icelan

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [7]:
# Year to which data is formating to:
data_target_year = '2030'

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    5. Availability Factors Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
<br>
The dispa-SET technologies nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.1. Dispa-SET Time Step
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The time series data must be resampled to a predetermined time step.
<br>
The UNLEASH project utilizes three levels of granularity
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
One hour (1h) 
<li>
Thirty minutes (30min)
<li>
Fifteen minutes (15min)
</li>
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [8]:
# Time step to which data is formating to:
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.2. Empty Zone DataFrames
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating empty dataframes—column headers only—for the selected zones, corresponding to the target year
<hr style="border: 1px solid skyblue;">
</div>

In [9]:
# convert data year to integer
data_target_year = int(data_target_year)  

# Dictionary to store created DataFrames
availability_factors_dfs_dict = {}

for zone in zone_names:
    zone_folder = os.path.join(availability_factors_base_data_folder_path, zone)
    if not os.path.isdir(zone_folder):
        continue  # skip if zone folder doesn't exist

    # Go into subfolder matching data_target_time_step
    timestep_folder = os.path.join(zone_folder, str(data_target_time_step))
    if not os.path.isdir(timestep_folder):
        continue  # skip if subfolder doesn't exist

    # List all csv files in the timestep folder
    csv_files = [f for f in os.listdir(timestep_folder) if f.endswith(".csv")]

    # Extract years from filenames (must be digits only)
    available_years = []
    for f in csv_files:
        name, ext = os.path.splitext(f)
        if name.isdigit():  # e.g., "2004"
            available_years.append(int(name))

    if not available_years:
        continue  # skip if no year-based CSVs exist

    # Find closest year to target
    closest_year = min(available_years, key=lambda y: abs(y - data_target_year))

    # Path to the chosen file
    chosen_file = os.path.join(timestep_folder, f"{closest_year}.csv")

    # Read only the first column
    first_col_name = pd.read_csv(chosen_file, nrows=0).columns[0]  # get first column name
    first_col_data = pd.read_csv(chosen_file, usecols=[first_col_name])

    # Read all headers
    all_headers = pd.read_csv(chosen_file, nrows=0).columns.tolist()

    # Create new DataFrame: first column has data, others are empty
    df = pd.DataFrame(columns=all_headers)
    df[first_col_name] = first_col_data[first_col_name]

    # Store in dictionary and optionally as global variable
    df_name = f"{zone}_{data_target_year}"
    availability_factors_dfs_dict[df_name] = df
    globals()[df_name] = df

    print(f"Zone {zone}: picked {closest_year}.csv for target {data_target_year} and copied first column")

print("Availability Factors Data Frames:", list(availability_factors_dfs_dict.keys()))

Zone BE: picked 2023.csv for target 2030 and copied first column
Zone FR: picked 2023.csv for target 2030 and copied first column
Zone DE: picked 2023.csv for target 2030 and copied first column
Zone NL: picked 2023.csv for target 2030 and copied first column
Zone UK: picked 2023.csv for target 2030 and copied first column
Availability Factors Data Frames: ['BE_2030', 'FR_2030', 'DE_2030', 'NL_2030', 'UK_2030']


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.3. Time Step Correction
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Verifying and correcting first column timestamps in each zone dataframe to match target year and selected time step.
<hr style="border: 1px solid skyblue;">
</div>

In [10]:
# Define a mapping for time step strings to pandas frequency strings
time_step_map = {
    
    "1h": "H",
    "15min": "15T",
    "30min": "30T"

}

# Get the pandas frequency string for the target time step
freq = time_step_map[data_target_time_step]

for df_name, df in availability_factors_dfs_dict.items():
    if df.empty:
        continue  # skip empty dataframes
    
    # Identify the first column
    first_col = df.columns[0]
    
    # Convert column to datetime with UTC if not already
    df[first_col] = pd.to_datetime(df[first_col], utc=True, errors='coerce')

    # Remove any rows that failed to parse
    df = df.dropna(subset=[first_col])

    # Create a date range for the correct year and time step
    start_time = pd.Timestamp(f"{data_target_year}-01-01 00:00:00", tz="UTC")
    end_time = pd.Timestamp(f"{data_target_year}-12-31 23:59:59", tz="UTC")

    correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)

    # Replace the first column with the corrected date range
    if len(correct_index) >= len(df):
        df[first_col] = correct_index[:len(df)]
    else:
        # If df has more rows than the date range, extend with repeated values
        repeats = (len(df) // len(correct_index)) + 1
        df[first_col] = pd.Series(list(correct_index) * repeats)[:len(df)]

    # Update the DataFrame in the dictionary
    availability_factors_dfs_dict[df_name] = df
    globals()[df_name] = df  # optional if you use global variables

    print(f"Updated {df_name}: first column aligned with {data_target_year} and {data_target_time_step}")

Updated BE_2030: first column aligned with 2030 and 1h
Updated FR_2030: first column aligned with 2030 and 1h
Updated DE_2030: first column aligned with 2030 and 1h
Updated NL_2030: first column aligned with 2030 and 1h
Updated UK_2030: first column aligned with 2030 and 1h


/tmp/ipykernel_971561/1282086355.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_971561/1282086355.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_971561/1282086355.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_971561/1282086355.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_971561/1282086355.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [17]:
print (f"Name of the DispaSET Unleash folder:                          {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                          {dispaSET_unleash_folder_path}\n")
print (f"Name of the Availability Factors Base data folder:            {availability_factors_base_data_folder_name}\n")
print (f"Path to the Availability Factors Base data folder:            {availability_factors_base_data_folder_path}\n")
print (f"Name of Availability Factors_Pypsa Raw data folder:           {availability_factors_pypsa_raw_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Raw data folder:       {availability_factors_pypsa_raw_data_folder_path}\n")
print (f"Name of the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                            {zone_names}\n")
print (f"Name of the zone_names_equivalences_dict (dictionary):        {list(zone_names_equivalences_dict.keys())}\n")
print (f"Target year:                                                  {data_target_year}\n")
print (f"Target time step:                                             {data_target_time_step}\n")
print (f"Name of the Availability Factors DataFrames (dictionary):     {list(availability_factors_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                          Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                          /home/ray/Dispa-SET_Unleash

Name of the Availability Factors Base data folder:            AvailabilityFactors

Path to the Availability Factors Base data folder:            /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors

Name of Availability Factors_Pypsa Raw data folder:           dispatch

Path to the Availability Factors_Pypsa Raw data folder:       /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/dispatch

Name of the Availability Factors_Pypsa Formated data folder:  AvailabilityFactors

Path to the Availability Factors_Pypsa Formated data folder:  /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/AvailabilityFactors

Name of the zones:                                            ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dict (dictionary):        ['AL', 'AM', 'AT', 'AZ', 'BY', 'BE',

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
6. Nomenclature Technology Dictionary creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary provides the mapping—or cross-reference—of renewable source technology names between the PyPSA and Dispa-SET modeling frameworks.
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.1. Dispa-SET Technologies Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The dispa-SET technologies nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div> 

In [8]:
# Define all the Dispa-SET technology lists from the common.py script of Dispa-SET core scripts
tech_master_list =         []

tech_renewables =          ['HROR' , 'PHOT' , 'WAVE' , 'WTOF' , 'WTON' , 'SOTH']

tech_conventional =        ['HDAM' , 'COMC' , 'GTUR' , 'STUR' , 'BATS' , 'ICEN']

tech_batteries =           ['BATS']

tech_storage =             ['BATS' , 'HDAM' , 'HPHS' , 'BEVS' , 'CAES' , 'SCSP' , 'H2ST' , 'HPHSC', 'THMS']

tech_p2bs =                ['P2GS' , 'ALKE' , 'PEME' , 'SOXE' , 'P2BS' , 'PEFC' , 'DMFC' , 'ALFC' , 'PAFC' , 'MCFC' , 'SOFC' ,
                            'REFC' , 'HDAMC', 'HRORC', 'HDLZ' , 'COMCX', 'GTURX', 'ICENX', 'STURX', 'P2HT' , 'ASHP' , 'GSHP' , 
                            'HYHP' , 'WSHP' , 'REHE']

tech_bs2p =                ['BSPG']

tech_boundary_sector =     ['BSPG' , 'GETH' , 'HOBO' , 'SOTH' , 'ABHP' , 'HOBOX', 'P2BS' , 'HBBS' , 'WHEN']

# Combine all lists into a single set to get unique values
all_technologies_set = set(tech_master_list + tech_renewables + tech_conventional +
                           tech_batteries + tech_storage + tech_p2bs + tech_bs2p +
                           tech_boundary_sector)

# Convert the set back to a sorted list
all_technologies_list = sorted(list(all_technologies_set))

# Create a DataFrame with the single column
dispaSET_tech_list = pd.DataFrame(all_technologies_list, columns=['Dispa-SET Technologies'])

# Print the final DataFrame
dispaSET_tech_list

,Dispa-SET Technologies
0,ABHP
1,ALFC
2,ALKE
3,ASHP
4,BATS
5,BEVS
6,BSPG
7,CAES
8,COMC
9,COMCX


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.2. Dispa-SET Fuels Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Additionally the dispa-SET fuelss nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [9]:
# Define all the Dispa-SET fuel lists from the common.py script of Dispa-SET core scripts
dispaSET_fuel_list =      [ 'AIR', 'AMO', 'BIO', 'GAS', 'HRD', 'LIG', 'NUC', 'OIL', 'PEA', 'SUN', 
                            'WAT', 'WIN', 'WST', 'OTH', 'GEO', 'HYD', 'WHT', 'ELE', 'THE', 'UNK'  ]

# Create a DataFrame with the single column
dispaSET_fuel_list = pd.DataFrame(dispaSET_fuel_list, columns=['Dispa-SET Fuels'])

# Print the final DataFrame
dispaSET_fuel_list

,Dispa-SET Fuels
0,AIR
1,AMO
2,BIO
3,GAS
4,HRD
5,LIG
6,NUC
7,OIL
8,PEA
9,SUN


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.3. Technologies & Demmands Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.3. Dispa-SET vs PyPSA Technologies Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
To facilitate data harmonization, a dictionary containing the Dispa-SET and PyPSA technology equivalences for renewable power plant units is developed, leveraging the specifications detailed in the preceding table.
</div>
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 12px; font-family: TimesNewRoman; color:skyblue">
    * Notes: &nbsp;&nbsp; Keep values of the first column identical to the 'dispaSET_tech_list' list, since it depends of the Dipsa-SET core code.
    <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    If the list in the <code>commons.py</code> script changes, the 'tech_equivalences' list has to be updated accordingly.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [42]:
# Dictionary mapping Dispa-SET acronyms to PyPSA tech names

tech_equivalences_dict = {
    
# -----------------------
# Renewable Power Units
# -----------------------
"WTON"  :  {"tech": ["onshore wind"       ] ,   "curt": ["onshore curtailment"    ]},
    
"WTOF"  :  {"tech": ["offshore wind"      ] ,   "curt": ["offshore curtailment"   ]},
    
"PHOT"  :  {"tech": ["solar"              ] ,   "curt": ["solar curtailment"      ]},
    
"HROR"  :  {"tech": ["hydroelectricity"   ] ,   "curt": [" "   ]},
       
}

tech_equivalences_dict

{'WTON': {'tech': ['onshore wind'], 'curt': ['onshore curtailment']},
 'WTOF': {'tech': ['offshore wind'], 'curt': ['offshore curtailment']},
 'PHOT': {'tech': ['solar'], 'curt': ['solar curtailment']},
 'HROR': {'tech': ['hydroelectricity'], 'curt': [' ']}}

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [18]:
print (f"Name of the DispaSET Unleash folder:                          {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                          {dispaSET_unleash_folder_path}\n")
print (f"Name of the Availability Factors Base data folder:            {availability_factors_base_data_folder_name}\n")
print (f"Path to the Availability Factors Base data folder:            {availability_factors_base_data_folder_path}\n")
print (f"Name of Availability Factors_Pypsa Raw data folder:           {availability_factors_pypsa_raw_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Raw data folder:       {availability_factors_pypsa_raw_data_folder_path}\n")
print (f"Name of the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                            {zone_names}\n")
print (f"Name of the zone_names_equivalences_dict (dictionary):        {list(zone_names_equivalences_dict.keys())}\n")
print (f"Target year:                                                  {data_target_year}\n")
print (f"Target time step:                                             {data_target_time_step}\n")
print (f"Name of the Availability Factors DataFrames (dictionary):     {list(availability_factors_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:             tech_equivalences_dict\n")

Name of the DispaSET Unleash folder:                          Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                          /home/ray/Dispa-SET_Unleash

Name of the Availability Factors Base data folder:            AvailabilityFactors

Path to the Availability Factors Base data folder:            /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors

Name of Availability Factors_Pypsa Raw data folder:           dispatch

Path to the Availability Factors_Pypsa Raw data folder:       /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/dispatch

Name of the Availability Factors_Pypsa Formated data folder:  AvailabilityFactors

Path to the Availability Factors_Pypsa Formated data folder:  /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/AvailabilityFactors

Name of the zones:                                            ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dict (dictionary):        ['AL', 'AM', 'AT', 'AZ', 'BY', 'BE',

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    7. Raw Data Uploading
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The availability factor time series will be constructed from the hourly PyPSA dispatch values for each energy carrier
    <br>
 These values will be stored in a dictionary partitioned by country.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [22]:
# Initialize the dictionary to store DataFrames
nodal_dispatch_dfs_dict = {}

# Loop through each country acronym in zone_names
for zone in zone_names:
    # Try the primary file name
    primary_file_name = f"{zone}_dispatch.csv"
    primary_file_path = os.path.join(availability_factors_pypsa_raw_data_folder_path, primary_file_name)

    # If the primary file exists, read it
    if os.path.exists(primary_file_path):
        df = pd.read_csv(primary_file_path)
        nodal_dispatch_dfs_dict[f"{zone}_dispatch_df"] = df
    else:
        # If the primary file does not exist, try alternative acronyms
        alternative_acronyms = zone_names_equivalences_dict.get(zone, {}).get("Acronym", [])
        for alt_acronym in alternative_acronyms:
            alternative_file_name = f"{alt_acronym}_dispatch.csv"
            alternative_file_path = os.path.join(availability_factors_pypsa_raw_data_folder_path, alternative_file_name)
            if os.path.exists(alternative_file_path):
                df = pd.read_csv(alternative_file_path)
                nodal_dispatch_dfs_dict[f"{zone}_dispatch_df"] = df
                break  # Stop after the first successful alternative
        else:
            print(f"No file found for zone: {zone} (tried: {primary_file_name}, alternatives: {[f'{a}_dispatch.csv' for a in alternative_acronyms]})")

nodal_dispatch_dfs_dict

{'BE_dispatch_df':                Unnamed: 0      CCGT       CHP       DAC  Imports_Exports  \
 0     2013-01-01 00:00:00  0.000001  1.001630 -0.060034         4.599342   
 1     2013-01-01 01:00:00  0.000001  1.001630 -0.060034         4.747412   
 2     2013-01-01 02:00:00  0.000001  1.001630 -0.060034         5.105037   
 3     2013-01-01 03:00:00  0.000001  1.001630 -0.060034         5.522517   
 4     2013-01-01 04:00:00  0.000001  1.001631 -0.060034         1.466305   
 ...                   ...       ...       ...       ...              ...   
 8755  2013-12-31 19:00:00  0.000001  1.001630 -0.060034         3.561013   
 8756  2013-12-31 20:00:00  0.000001  1.001630 -0.060034         3.561012   
 8757  2013-12-31 21:00:00  0.000001  1.001630 -0.060034         3.561010   
 8758  2013-12-31 22:00:00  0.000001  1.001630 -0.060034         5.938913   
 8759  2013-12-31 23:00:00  0.000001  1.001630 -0.060034         6.464819   
 
               OCGT  battery storage  distribution netwo

In [24]:
import pandas as pd

# Paths to the two files
file1_path = "/home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/dispatch/GB_dispatch.csv"
file2_path = "/home/ray/Downloads/Downloads/Test/UK_dispatch_df_2.csv"

# Read the files into DataFrames
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# Compare the DataFrames
are_equal = df1.equals(df2)

# Print the result
if are_equal:
    print("The two files have exactly the same content.")
else:
    print("The two files do NOT have the same content.")

    # Optionally, show differences (if you want to debug)
    print("\nDifferences:")
    print("File 1 shape:", df1.shape)
    print("File 2 shape:", df2.shape)
    print("\nFirst few rows of File 1:")
    print(df1.head())
    print("\nFirst few rows of File 2:")
    print(df2.head())


The two files do NOT have the same content.

Differences:
File 1 shape: (8760, 16)
File 2 shape: (8760, 16)

First few rows of File 1:
            Unnamed: 0      CCGT       CHP       DAC  Imports_Exports  \
0  2013-01-01 00:00:00  0.000002  0.193057 -0.099578       -32.613636   
1  2013-01-01 01:00:00  0.000002  0.193058 -0.099578       -32.613719   
2  2013-01-01 02:00:00  0.000002  0.193058 -0.099578       -32.608090   
3  2013-01-01 03:00:00  0.000002  0.193058 -0.099578       -32.607074   
4  2013-01-01 04:00:00  0.000002  0.193058 -0.099578       -32.607531   

   battery storage  distribution network  hydroelectricity   nuclear  \
0         4.325817            -55.734911          1.387101  8.195564   
1         2.273065            -56.578228          1.388357  8.195565   
2        -0.954155            -55.891088          1.170835  7.576665   
3        -0.870649            -54.947815          1.178327  7.600293   
4        -0.831053            -59.189166          1.166430  7.6041

In [23]:
# Define the path where the CSV files will be saved
export_path = "/home/ray/Downloads/Downloads/Test/"

# Ensure the directory exists
os.makedirs(export_path, exist_ok=True)

# Loop through each DataFrame in the dictionary
for country, df in nodal_dispatch_dfs_dict.items():
    # Define the full path for each CSV file
    filename = os.path.join(export_path, f"{country}_2.csv")

    # Export the DataFrame to CSV
    df.to_csv(filename, index=False)

    print(f"Exported {country} to {filename}")


Exported BE_dispatch_df to /home/ray/Downloads/Downloads/Test/BE_dispatch_df_2.csv
Exported FR_dispatch_df to /home/ray/Downloads/Downloads/Test/FR_dispatch_df_2.csv
Exported DE_dispatch_df to /home/ray/Downloads/Downloads/Test/DE_dispatch_df_2.csv
Exported NL_dispatch_df to /home/ray/Downloads/Downloads/Test/NL_dispatch_df_2.csv
Exported UK_dispatch_df to /home/ray/Downloads/Downloads/Test/UK_dispatch_df_2.csv


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Setting the target year row as columns index.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [61]:
target_int = int(data_target_year)

# --- Step 1: Convert row to numeric where possible, check for match ---
def row_contains_target_year(row):
    # convert row to numeric where possible
    numeric_row = pd.to_numeric(row, errors='coerce')
    if (numeric_row == target_int).any():
        return True
    # fallback: check string equality
    str_row = row.astype(str).str.strip()
    if (str_row == data_target_year).any():
        return True
    return False

# Find the first matching row
first_year_row_index = nodal_capacities_df.apply(row_contains_target_year, axis=1).idxmax()

# --- Step 2: Use that row as new headers ---
new_columns = nodal_capacities_df.loc[first_year_row_index].astype(str).tolist()

# --- Step 3: Drop the row and set new headers ---
df_with_new_headers = nodal_capacities_df.drop(first_year_row_index).copy()
df_with_new_headers.columns = new_columns
nodal_capacities_df = df_with_new_headers.reset_index(drop=True)

nodal_capacities_df

,planning_horizon,nan,nan,2030,2040,2050
0,ll,NaN,NaN,vopt,vopt,vopt
1,opt,NaN,NaN,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1
2,generators,NaN,coal,33028.7100662858,20.157946290498,0.00439350191459953
3,generators,NaN,lignite,11143.9976777995,19.0295991828038,0.00367573147029184
4,generators,NaN,load,1000000,1000000,1000000
...,...,...,...,...,...,...
651,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
652,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
653,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
654,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Naming undefined columns.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [62]:
def is_unnamed_label(label):
    """Return True for NaN/empty/'nan'/'Unnamed...' column labels."""
    if pd.isna(label):
        return True
    s = str(label).strip()
    if s == '' or s.lower() == 'nan':
        return True
    if re.match(r'Unnamed', s):
        return True
    return False

# Assuming nodal_capacities_df is the DataFrame and zone_names is defined

cols = list(nodal_capacities_df.columns)
new_cols = []
zone_count = 0
tech_count = 0

for i, col_label in enumerate(cols):
    if is_unnamed_label(col_label):
        # read the column by position to avoid issues with NaN labels
        col_series = nodal_capacities_df.iloc[:, i].dropna().astype(str)

        # find any zone substring (case-sensitive)
        found_zone = False
        for val in col_series:
            # skip obvious 'nan' strings
            if val.strip().lower() == 'nan':
                continue
            for zone in zone_names:
                if zone in val:      # case-sensitive substring match
                    found_zone = True
                    break
            if found_zone:
                break

        if found_zone:
            zone_count += 1
            new_name = 'zone' if zone_count == 1 else f'zone_{zone_count}'
        else:
            tech_count += 1
            new_name = 'tech' if tech_count == 1 else f'tech_{tech_count}'

        new_cols.append(new_name)
    else:
        new_cols.append(col_label)

# apply the new column names determined above
nodal_capacities_df.columns = new_cols

# Change the header of the first column (index 0) to 'Element'
nodal_capacities_df.columns.values[0] = 'Element'
# --- NEW CODE ADDITION END ---

# quick check
print(list(zip(cols, nodal_capacities_df.columns)))

nodal_capacities_df

[('planning_horizon', 'Element'), ('nan', 'zone'), ('nan', 'tech'), ('2030', '2030'), ('2040', '2040'), ('2050', '2050')]


,Element,zone,tech,2030,2040,2050
0,ll,NaN,NaN,vopt,vopt,vopt
1,opt,NaN,NaN,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1,1H-T-H-B-I-A-dist1
2,generators,NaN,coal,33028.7100662858,20.157946290498,0.00439350191459953
3,generators,NaN,lignite,11143.9976777995,19.0295991828038,0.00367573147029184
4,generators,NaN,load,1000000,1000000,1000000
...,...,...,...,...,...,...
651,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
652,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
653,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
654,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering all the rows not related with the selected countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [63]:
# Function to check if a row contains any zone substring
def row_contains_zone(row, zones):
    for val in row.astype(str):
        for zone in zones:
            if zone in val:  # case-sensitive substring match
                return True
    return False

# Filter rows
mask = nodal_capacities_df.apply(lambda row: row_contains_zone(row, zone_names), axis=1)
nodal_capacities_df = nodal_capacities_df[mask].reset_index(drop=True)

nodal_capacities_df

,Element,zone,tech,2030,2040,2050
0,generators,BE1 0,gas,39563.0509463557,39563.0509463557,39563.0509463557
1,generators,BE1 0,load,31000000,30000000,28000000
2,generators,BE1 0,offwind,2261.8,2261.8,1549.8
3,generators,BE1 0,offwind-ac,1738.19937359281,1738.19937359281,1738.19937359281
4,generators,BE1 0,offwind-dc,1999.99944646756,4000.00050712006,4712.00052697313
...,...,...,...,...,...,...
626,stores,NL1 0,residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
627,stores,NL1 0,services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
628,stores,NL1 0,services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
629,stores,NL1 0,solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Combining the columnes 'Element' and 'tech' to adequate with the Technologies Dictionary nomenclature.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [64]:
# Join 'Element' and 'tech' with a space
nodal_capacities_df['tech'] = nodal_capacities_df['Element'] + ' ' + nodal_capacities_df['tech']

# Drop the 'Element' column
nodal_capacities_df = nodal_capacities_df.drop(columns=['Element'])

nodal_capacities_df

,zone,tech,2030,2040,2050
0,BE1 0,generators gas,39563.0509463557,39563.0509463557,39563.0509463557
1,BE1 0,generators load,31000000,30000000,28000000
2,BE1 0,generators offwind,2261.8,2261.8,1549.8
3,BE1 0,generators offwind-ac,1738.19937359281,1738.19937359281,1738.19937359281
4,BE1 0,generators offwind-dc,1999.99944646756,4000.00050712006,4712.00052697313
...,...,...,...,...,...
626,NL1 0,stores residential urban decentral water tanks,0.343476644057879,1.37948036679268,1.4466688695137
627,NL1 0,stores services rural water tanks,1.7106336080301,4.92764035982957,4.9974419298418
628,NL1 0,stores services urban decentral water tanks,0.409615202103847,0.786612903715892,0.852507027902251
629,NL1 0,stores solid biomass,17227013.3536155,19944981.651406,22871222.6749819


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering the target year.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [65]:
# list of columns to keep
cols_to_keep = [col for col in nodal_capacities_df.columns 
                if str(col).startswith("zone") 
                or str(col).startswith("tech") 
                or str(col) == str(data_target_year)]

# filter dataframe
nodal_capacities_df = nodal_capacities_df[cols_to_keep].copy()

nodal_capacities_df

,zone,tech,2030
0,BE1 0,generators gas,39563.0509463557
1,BE1 0,generators load,31000000
2,BE1 0,generators offwind,2261.8
3,BE1 0,generators offwind-ac,1738.19937359281
4,BE1 0,generators offwind-dc,1999.99944646756
...,...,...,...
626,NL1 0,stores residential urban decentral water tanks,0.343476644057879
627,NL1 0,stores services rural water tanks,1.7106336080301
628,NL1 0,stores services urban decentral water tanks,0.409615202103847
629,NL1 0,stores solid biomass,17227013.3536155


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left:0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Cleaning the zone names.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [66]:
def replace_with_zone(cell, zones):
    cell_str = str(cell)  # ensure string
    for zone in zones:
        if zone in cell_str:  # case-sensitive substring match
            return zone       # replace whole cell with the matched zone
    return cell  # keep original if no match

# Apply only to the 'zone' column
nodal_capacities_df['zone'] = nodal_capacities_df['zone'].apply(lambda x: replace_with_zone(x, zone_names))

nodal_capacities_df

,zone,tech,2030
0,BE,generators gas,39563.0509463557
1,BE,generators load,31000000
2,BE,generators offwind,2261.8
3,BE,generators offwind-ac,1738.19937359281
4,BE,generators offwind-dc,1999.99944646756
...,...,...,...
626,NL,stores residential urban decentral water tanks,0.343476644057879
627,NL,stores services rural water tanks,1.7106336080301
628,NL,stores services urban decentral water tanks,0.409615202103847
629,NL,stores solid biomass,17227013.3536155


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Dividing the capacities file into the selected countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [67]:
# Create dictionary to store the derived DataFrames
nodal_capacities_dfs_dict = {}

for zone in zone_names:
    # Filter rows where 'zone' matches the current zone
    df_zone = nodal_capacities_df[nodal_capacities_df['zone'] == zone].copy()
    
    # Save into dictionary with a descriptive key
    nodal_capacities_dfs_dict[f"{zone}_nodal_capacities_df"] = df_zone

nodal_capacities_dfs_dict

{'BE_nodal_capacities_df':     zone                                            tech               2030
 0     BE                                  generators gas   39563.0509463557
 1     BE                                 generators load           31000000
 2     BE                              generators offwind             2261.8
 3     BE                           generators offwind-ac   1738.19937359281
 4     BE                           generators offwind-dc   1999.99944646756
 ..   ...                                             ...                ...
 561   BE  stores residential urban decentral water tanks  0.620639625874326
 562   BE               stores services rural water tanks   1.00436547503266
 563   BE     stores services urban decentral water tanks  0.680430055032749
 564   BE                            stores solid biomass   20944781.4561726
 565   BE                stores urban central water tanks   714893.437595446
 
 [106 rows x 3 columns],
 'FR_nodal_capacities_d

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Aggregating repeated tech items.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [68]:
# Convert the integer year to a string to match the column name
data_target_year_str = str(data_target_year)

# Create an empty DataFrame to store the rows that will be erased
erased_rows_df = pd.DataFrame()

for df_name in nodal_capacities_dfs_dict:
    df = nodal_capacities_dfs_dict[df_name]

    # Convert the target year column to numeric, ignoring the header
    df[data_target_year_str] = pd.to_numeric(df[data_target_year_str], errors='coerce')

    # Identify duplicate rows based on 'zone' and 'tech', keeping the first occurrence
    duplicates = df.duplicated(subset=['zone', 'tech'], keep='first')

    # Select the rows that are duplicates (i.e., those to be erased)
    erased_part = df[duplicates].copy()

    # Append these erased rows to the new DataFrame
    erased_rows_df = pd.concat([erased_rows_df, erased_part], ignore_index=True)

    # Group by 'zone' and 'tech', then sum the target year column
    grouped_df = df.groupby(['zone', 'tech'], as_index=False)[data_target_year_str].sum()

    # Update the dictionary with the processed DataFrame
    nodal_capacities_dfs_dict[df_name] = grouped_df

# Verify the result
print("\n--- Processed DataFrames ---")
for df_name, df in nodal_capacities_dfs_dict.items():
    print(f"\nDataFrame: {df_name}")
    print(df)

print("\n--- Erased Rows DataFrame ---")
print(erased_rows_df)


--- Processed DataFrames ---

DataFrame: BE_nodal_capacities_df
    zone                                            tech          2030
0     BE                                  generators gas  3.956305e+04
1     BE                                 generators load  3.100000e+07
2     BE                              generators offwind  2.261800e+03
3     BE                           generators offwind-ac  1.738199e+03
4     BE                           generators offwind-dc  1.999999e+03
..   ...                                             ...           ...
101   BE  stores residential urban decentral water tanks  6.206396e-01
102   BE               stores services rural water tanks  1.004365e+00
103   BE     stores services urban decentral water tanks  6.804301e-01
104   BE                            stores solid biomass  2.094478e+07
105   BE                stores urban central water tanks  7.148934e+05

[106 rows x 3 columns]

DataFrame: FR_nodal_capacities_df
    zone                

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [69]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:          tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:       overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:              fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary):{list(nodal_capacities_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                               2030

Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    8. PyPSA to Dispa-SET Power Plants Data Formatting
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The power plants dataframe start to be formatting.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.1. Unit, PowerCapacity, Nunits and Zone Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The columns Unit, PowerCapacity, Nunits, Zone, Technology and Fuel of the final Power Plants inputs are added to the final Power Plant sheet.
    <br>
    Just the Unit, PowerCapacity, Nunits, Zone are correctly filtered at this stage.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [70]:
for zone in zone_names:
    # Construct keys for both dictionaries
    nodal_key = f"{zone}_nodal_capacities_df"
    power_key = f"{zone}_{data_target_year}"

    # Get the corresponding dataframes
    nodal_df = nodal_capacities_dfs_dict.get(nodal_key)
    power_df = power_plants_dfs_dict.get(power_key)

    if nodal_df is not None and power_df is not None:
        new_rows = []

        for _, row in nodal_df.iterrows():
            tech = row['tech']
            # Check if tech is in the equivalence dictionary
            if tech in tech_equivalences_dict:
                # Safely access the column using .loc
                power_capacity = row.loc[str(data_target_year)]

                # Create a new row with the required values
                new_row = {
                    'Unit': tech,
                    'PowerCapacity': power_capacity,
                    'Nunits': 1,
                    'Zone': row['zone'],
                    # MODIFIED LINES BELOW:
                    'Technology': ' '.join(tech_equivalences_dict[tech]['tech']) if isinstance(tech_equivalences_dict[tech]['tech'], list) else tech_equivalences_dict[tech]['tech'],
                    'Fuel': ' '.join(tech_equivalences_dict[tech]['fuel']) if isinstance(tech_equivalences_dict[tech]['fuel'], list) else tech_equivalences_dict[tech]['fuel']
                }

                new_rows.append(new_row)

        # Concatenate new rows to the existing dataframe
        if new_rows:
            power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)

        # Update the dictionary with the modified dataframe
        power_plants_dfs_dict[power_key] = power_df

power_plants_dfs_dict

/tmp/ipykernel_844689/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)
/tmp/ipykernel_844689/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_df = pd.concat([power_df, pd.DataFrame(new_rows)], ignore_index=True)
/tmp/ipykernel_844689/3897183125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no lon

{'BE_2030':    Unnamed: 0                                      Unit  PowerCapacity Nunits  \
 0         NaN                        generators offwind    2261.800000      1   
 1         NaN                     generators offwind-ac    1738.199374      1   
 2         NaN                     generators offwind-dc    1999.999446      1   
 3         NaN                         generators onwind    5999.998849      1   
 4         NaN                            generators ror      59.015340      1   
 5         NaN                          generators solar   11999.999412      1   
 6         NaN                  generators solar rooftop    1999.996368      1   
 7         NaN                                links CCGT    6279.540144      1   
 8         NaN                                 links DAC       0.007254      1   
 9         NaN                     links H2 Electrolysis     150.000000      1   
 10        NaN                        links H2 Fuel Cell     126.632530      1   
 11  

<div style="background-color: black;">
    <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.2. Technology Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The column technology is separating for all those units that can be matched with more than one technology.
    <br>
    The desegregation of the power capacity will be done according the following rule / proportion.
    </div>
    <div style="text-align: center; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color:skyblue; margin-top: 10px; margin-bottom: 10px;">
    $E_k = \frac{2^{N-k}}{2^N - 1}$
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Where $k$ is the position of the element / Technology, $k=1$ is the first, $k=N$ is the last
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [71]:
# Function to compute proportional weights
def compute_proportions(n):
    weights = [2**(n - k - 1) for k in range(n)]
    total = sum(weights)
    return [w / total for w in weights]

# Process each DataFrame in the dictionary
for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = power_plants_dfs_dict.get(key)

    if df is not None:
        new_rows = []

        for _, row in df.iterrows():
            techs = str(row['Technology']).split()
            if len(techs) > 1:
                proportions = compute_proportions(len(techs))
                for idx, (tech, prop) in enumerate(zip(techs, proportions), start=1):
                    new_row = row.copy()
                    new_row['Technology'] = tech
                    new_row['Unit'] = f"{row['Unit']}_{idx}"
                    new_row['PowerCapacity'] = row['PowerCapacity'] * prop
                    new_rows.append(new_row)
            else:
                new_rows.append(row)

        # Replace the original DataFrame with the expanded one
        power_plants_dfs_dict[key] = pd.DataFrame(new_rows)

power_plants_dfs_dict

{'BE_2030':     Unnamed: 0                                        Unit  PowerCapacity  \
 0          NaN                          generators offwind    2261.800000   
 1          NaN                       generators offwind-ac    1738.199374   
 2          NaN                       generators offwind-dc    1999.999446   
 3          NaN                           generators onwind    5999.998849   
 4          NaN                              generators ror      59.015340   
 5          NaN                            generators solar   11999.999412   
 6          NaN                    generators solar rooftop    1999.996368   
 7          NaN                                  links CCGT    6279.540144   
 8          NaN                                   links DAC       0.007254   
 9          NaN                     links H2 Electrolysis_1      85.714286   
 9          NaN                     links H2 Electrolysis_2      42.857143   
 9          NaN                     links H2 Electrol

<div style="background-color: black;">
    <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    8.3. Fuel Data Clasification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The column Feul is separating for all those units that can be matched with more than one fuel.
    <br>
    The desegregation of the power capacity will be done according the proportional technology - fuel tables for the corresponding country .
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [72]:
processed_power_plants_dfs_dict = {}

for zone in zone_names:
    power_key = f"{zone}_{data_target_year}"
    fuel_key = zone
    power_df = power_plants_dfs_dict.get(power_key)
    fuel_df = fuel_technologies_match_dict.get(fuel_key, overal_fuel_technologies_match_df)

    if power_df is None:
        print(f"⚠️ No power plant DataFrame found for {power_key}")
        continue

    df = power_df.copy()
    new_rows = []
    rows_to_drop = []

    for idx, row in df.iterrows():
        tech = str(row["Technology"]).strip()
        fuel_str = str(row["Fuel"]).strip()
        fuels = fuel_str.split()

        if len(fuels) <= 1:
            continue

        if tech not in fuel_df.index:
            rows_to_drop.append(idx)
            continue

        original_capacity = row["PowerCapacity"]
        new_capacities = []
        new_rows_for_this_row = []

        for i, fuel in enumerate(fuels, start=1):
            if fuel not in fuel_df.columns:
                continue
            multiplier = fuel_df.loc[tech, fuel]
            if pd.isna(multiplier):
                continue
            new_row = row.copy()
            new_row["Unit"] = f"{row['Unit']}_{i}"
            new_row["Fuel"] = fuel
            new_row["PowerCapacity"] = original_capacity * multiplier
            new_rows_for_this_row.append(new_row)
            new_capacities.append(new_row["PowerCapacity"])

        if not new_rows_for_this_row:
            rows_to_drop.append(idx)
            continue

        # Sum the new capacities
        sum_new_capacities = sum(new_capacities)
        # If the sum is less than the original, add the difference to the largest new capacity
        if sum_new_capacities < original_capacity:
            max_capacity_idx = max(range(len(new_capacities)), key=lambda i: new_capacities[i])
            new_rows_for_this_row[max_capacity_idx]["PowerCapacity"] += (original_capacity - sum_new_capacities)

        new_rows.extend(new_rows_for_this_row)
        rows_to_drop.append(idx)

    df = df.drop(rows_to_drop, errors="ignore")
    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    processed_power_plants_dfs_dict[power_key] = df

print("✅ Processing complete. Updated DataFrames stored in 'processed_power_plants_dfs_dict'.")

processed_power_plants_dfs_dict

✅ Processing complete. Updated DataFrames stored in 'processed_power_plants_dfs_dict'.


{'BE_2030':     Unnamed: 0                                        Unit  PowerCapacity  \
 0          NaN                          generators offwind    2261.800000   
 1          NaN                       generators offwind-ac    1738.199374   
 2          NaN                       generators offwind-dc    1999.999446   
 3          NaN                           generators onwind    5999.998849   
 4          NaN                              generators ror      59.015340   
 5          NaN                            generators solar   11999.999412   
 6          NaN                    generators solar rooftop    1999.996368   
 7          NaN                                   links DAC       0.007254   
 8          NaN                     links H2 Electrolysis_1      85.714286   
 9          NaN                     links H2 Electrolysis_2      42.857143   
 10         NaN                     links H2 Electrolysis_3      21.428571   
 11         NaN                        links H2 Fuel 

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [73]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                 {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                 {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:            {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:       {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):           {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:          tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:       overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:              fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary):{list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the Processed Power Plant DataFrames (dictionary): {list(processed_power_plants_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                 PowerPlants

Path to the Power Plants Base data folder:                 /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                PowerPlants

Path to the Power Plants_Pypsa Raw data folder:            /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:       PowerPlants

Path to the Power Plants_Pypsa Formated data folder:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                               2030

Name of the Power Plant DataFrames (dictionary):           ['BE_2030', 'FR_2030', 'DE_

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Erasing rows whit nule Power Capacity.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [74]:
# Iterate over each key in the dictionary
for key in list(processed_power_plants_dfs_dict.keys()):
    # Filter out rows where 'PowerCapacity' is 0
    processed_power_plants_dfs_dict[key] = processed_power_plants_dfs_dict[key][
        processed_power_plants_dfs_dict[key]['PowerCapacity'] != 0
    ]

processed_power_plants_dfs_dict

{'BE_2030':     Unnamed: 0                                        Unit  PowerCapacity  \
 0          NaN                          generators offwind    2261.800000   
 1          NaN                       generators offwind-ac    1738.199374   
 2          NaN                       generators offwind-dc    1999.999446   
 3          NaN                           generators onwind    5999.998849   
 4          NaN                              generators ror      59.015340   
 5          NaN                            generators solar   11999.999412   
 6          NaN                    generators solar rooftop    1999.996368   
 7          NaN                                   links DAC       0.007254   
 8          NaN                     links H2 Electrolysis_1      85.714286   
 9          NaN                     links H2 Electrolysis_2      42.857143   
 10         NaN                     links H2 Electrolysis_3      21.428571   
 11         NaN                        links H2 Fuel 

<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
  <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman'; color: skyblue;">
    8.4. CHP Units Classification
  </div>
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    The CHP units in Dispaset have the following features:
  </div>
    
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 11px; font-family: 'Times New Roman'; color: skyblue;">
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">CHPType</span>
    : This defines the operational characteristic of the CHP plant. It determines how the plant can vary its electricity and heat output.
    <br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPPowerToHeat</span>
    : Also known as the Power-to-Heat Ratio; σ. It's the ratio of electrical power output to useful heat output.
$$ \sigma = \frac{\text{Power Output}}{\text{Heat Output}} $$
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPPowerLossFactor</span>
    : It quantifies the amount of electrical power lost per unit of increased heat extraction. It represents the trade-off, i.e., producing more heat—by extracting steam—results in a loss of potential electricity generation.
    <br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CHPMaxHeat</span>
    : The highest amount of useful thermal energy—in MWth—that the unit can produce, determined by the size of its heat exchangers and district heating connections.
  </div>

<hr style="border: 2px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.1. CHP Units Nomenclature - CHP Technology List
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The following technologies can be set as CHP units:
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">* Notes</span>
    The CHP technology list may be expanded to cover additional options, e.g. PEME, ALKE and other technologies.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [75]:
# Define all the Dispa-SET CHP technology lists from the common.py script of Dispa-SET core scripts
chp_technologies_set = ['COMC', 'COMCX', 'GTUR', 'GTURX', 'ICEN', 'ICENX', 'STUR', 'STURX', 'MCFC', 'PAFC', 'PEFC', 'SOFC']

# Convert the set back to a sorted list
chp_technologies_list = sorted(list(chp_technologies_set))

# Create a DataFrame with the single column
dispaSET_chp_tech_list = pd.DataFrame(chp_technologies_list, columns=['Dispa-SET CHP Technologies'])

# Print the final DataFrame
dispaSET_chp_tech_list

,Dispa-SET CHP Technologies
0,COMC
1,COMCX
2,GTUR
3,GTURX
4,ICEN
5,ICENX
6,MCFC
7,PAFC
8,PEFC
9,SOFC


<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.2. CHP Units Nomenclature - CHPType Dictionary
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The dispa-SET CHP technologies nomenclature are loaded from: 
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
    </span>
    <br>
    The same is homogenized by the dictionary built in the next cell:   
    <div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">* Notes</span>
        Keep values of the first column identical to the 'List of CHP types', since it depends of the Dipsa-SET core code.
    <br>
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue"></span>
        If the list in the <code>commons.py</code> script changes, the 'chp_type_dictionary' has to be updated accordingly.
    <br>
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue"></span>
        This equivalence dictionary is currently limited and requires updating with additional values.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [76]:
# Dictionary mapping CHP types to descriptions and equivalent technologies
chp_type_dict = {
    
"extraction"    :   {"Description": "Extraction Condensing Turbine; Can vary power/heat output flexibly",     "Equivalent_CHPType":  ["-"                                           ]}  ,
    
"back-pressure" :   {"Description": "Back-Pressure Turbine; Produces power and heat in a fixed ratio"   ,     "Equivalent_CHPType":  ["links urban central gas CHP"               ,
                                                                                                                                      "links urban central gas CHP CC"            , 
                                                                                                                                      "links urban central solid biomass CHP"     ,
                                                                                                                                      "links urban central solid biomass CHP CC"    ]}  ,
    
"p2h"           :   {"Description": "Power-to-Heat Unit; Converts electrical energy into heat"          ,     "Equivalent_CHPType":  ["-"                                           ]}

}

chp_type_dict

{'extraction': {'Description': 'Extraction Condensing Turbine; Can vary power/heat output flexibly',
  'Equivalent_CHPType': ['-']},
 'back-pressure': {'Description': 'Back-Pressure Turbine; Produces power and heat in a fixed ratio',
  'Equivalent_CHPType': ['links urban central gas CHP',
   'links urban central gas CHP CC',
   'links urban central solid biomass CHP',
   'links urban central solid biomass CHP CC']},
 'p2h': {'Description': 'Power-to-Heat Unit; Converts electrical energy into heat',
  'Equivalent_CHPType': ['-']}}

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Classifying the CHP unit type.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [77]:
for key, df in processed_power_plants_dfs_dict.items():
    for index, row in df.iterrows():
        if row['Technology'] in dispaSET_chp_tech_list['Dispa-SET CHP Technologies'].values:
            unit = row['Unit']
            for chp_key, chp_value in chp_type_dict.items():
                # Check if the unit starts with any of the values in Equivalent_CHPType
                if any(unit.startswith(equivalent) for equivalent in chp_value['Equivalent_CHPType']):
                    df.at[index, 'CHPType'] = chp_key
                    break

# Print the updated column from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPType column for {key}:")
    print(df[['CHPType']])

CHPType column for BE_2030:
          CHPType
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6             NaN
7             NaN
8             NaN
9             NaN
10            NaN
11            NaN
12            NaN
13            NaN
14            NaN
15            NaN
16            NaN
17            NaN
18            NaN
19            NaN
20            NaN
21            NaN
22            NaN
23            NaN
24            NaN
25  back-pressure
26  back-pressure
27  back-pressure
28  back-pressure
29  back-pressure
30  back-pressure
31  back-pressure
32  back-pressure
33  back-pressure
34            NaN
35            NaN
36            NaN
38            NaN
39            NaN
41            NaN
43            NaN
CHPType column for FR_2030:
          CHPType
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6             NaN
7             NaN
8             NaN
9       

/tmp/ipykernel_844689/3802790224.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_844689/3802790224.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_844689/3802790224.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'back-pressure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'CHPType'] = chp_key
/tmp/ipykernel_844689/3802790224.py:8: FutureWarning: Setting an item of inc

<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.4.3. CHP Units Nomenclature - CHP Features Dictionary
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The CHP features values—CHPPowerToHeat; CHPPowerLossFactor; CHPMaxHeat; COP—are updated into the dictionary from the following sources:
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    1. UROSEVIC, D. et al., 2013: "<a href="https://www.sciencedirect.com/science/article/pii/S0360544213005975" style="color:skyblue">https://www.sciencedirect.com/science/article/pii/S0360544213005975</a>"
    </span>
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    2. LECOMTE, T. et al., 2017: "<a href="https://publications.jrc.ec.europa.eu/repository/handle/JRC107769" style="color:skyblue">https://publications.jrc.ec.europa.eu/repository/handle/JRC107769</a>"
    </span>
    <br>
    <span style="display: inline-block; margin-left: 2.0em; font-size: 11px">
    3. EPA Catalog of CHP Technologies, 2017: "<a href="https://www.epa.gov/sites/default/files/2015-07/documents/catalog_of_chp_technologies.pdf" style="color:skyblue">https://www.epa.gov/sites/default/files/2015-07/documents/catalog_of_chp_technologies.pdf</a>"
    </span>
<hr style="border: 0.5px solid skyblue;">
</div>

In [78]:
# Dictionary mapping CHP technologies features 
chp_parameters_dict = pd.DataFrame({
    
    'Technology'         :   [  'COMC'                          , 'GTUR'                          , 'STUR'                          , 'ICEN'                          ,
                                'SOXE'                          , 'SOFC'                          , 'MCFC'                          , 'HBBS'                          ,
                                'COMC'                          , 'GTUR'                          , 'STUR'                          , 'ICEN'                          , 
                                'PEME'                          , 'ALKE'                          , 'PEFC'                          , 'PAFC'                          ,
                                'DMFC'                          , 'REFC'                          ,
                                'ASHP'                          , 'GSHP'                          , 'WSHP'                                                               ],
    
    'CHPType'            :   [  'extraction'                    , 'extraction'                    , 'extraction'                    , 'extraction'                    ,
                                'extraction'                    , 'extraction'                    , 'extraction'                    , 'extraction'                    ,
                                'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 , 
                                'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 , 'back-pressure'                 ,
                                'back-pressure'                 , 'back-pressure'                 ,
                                'p2h'                           , 'p2h'                           , 'p2h'                                                                ],
    
    'CHPPowerToHeat'     :   [  0.55                            , 0.45                            , 0.40                            , 0.5                             ,
                                0.20                            , 0.25                            , 0.20                            , 0.20                            ,
                                0.65                            , 0.55                            , 0.6                             , 0.45                            , 
                                0.30                            , 0.35                            , 0.40                            , 0.35                            ,
                                0.45                            , 0.40                            ,
                                None                            , None                            , None                            ,                                    ],
    
    'CHPPowerLossFactor' :   [  0.25                            , 0.35                            , 0.3                             , 0.2                             ,
                                0.15                            , 0.12                            , 0.08                            , 0.15                            ,
                                None                            , None                            , None                            , None                            , 
                                None                            , None                            , None                            , None                            ,
                                None                            , None                            ,
                                None                            , None                            , None                                                                 ],
    
    'CHPMaxHeat'         :   [  'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           , 'PowerCapacity * 1.2'           ,
                                'PowerCapacity * 1.5'           , 'PowerCapacity * 1.5'           , 'PowerCapacity * 1.8'           , 'PowerCapacity * 1.8'           ,
                                'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 
                                'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat',
                                'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat',
                                'PowerCapacity * COP'           , 'PowerCapacity * COP'           , 'PowerCapacity * COP'                                               ],
                                                                  
    'COP'                :   [  None                            , None                            , None                            , None                            ,
                                None                            , None                            , None                            , None                            ,
                                None                            , None                            , None                            , None                            , 
                                None                            , None                            , None                            , None                            ,
                                None                            , None                            ,
                                2.8                             , 3.5                             , 4.0                                                                  ]
})

chp_parameters_dict

,Technology,CHPType,CHPPowerToHeat,CHPPowerLossFactor,CHPMaxHeat,COP
0,COMC,extraction,0.55,0.25,PowerCapacity * 1.2,NaN
1,GTUR,extraction,0.45,0.35,PowerCapacity * 1.2,NaN
2,STUR,extraction,0.40,0.30,PowerCapacity * 1.2,NaN
3,ICEN,extraction,0.50,0.20,PowerCapacity * 1.2,NaN
4,SOXE,extraction,0.20,0.15,PowerCapacity * 1.5,NaN
5,SOFC,extraction,0.25,0.12,PowerCapacity * 1.5,NaN
6,MCFC,extraction,0.20,0.08,PowerCapacity * 1.8,NaN
7,HBBS,extraction,0.20,0.15,PowerCapacity * 1.8,NaN
8,COMC,back-pressure,0.65,NaN,PowerCapacity / CHPPowerToHeat,NaN
9,GTUR,back-pressure,0.55,NaN,PowerCapacity / CHPPowerToHeat,NaN


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Fullfilling the CHP features—CHPPowerToHeat; CHPPowerLossFactor; COP—from the CHP parameters dictionary.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [79]:
# Columns to bring from chp_parameters_dictionary
merge_cols = ['Technology'     , 'CHPType']
param_cols = ['CHPPowerToHeat' , 'CHPPowerLossFactor' , 'CHPMaxHeat' , 'COP']

for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = processed_power_plants_dfs_dict[key]

    # Keep only rows with a non-empty CHPType
    mask = df['CHPType'].notna() & (df['CHPType'] != "")

    # Merge only the rows where merge is needed
    df_merge = df.loc[mask, merge_cols].merge(
        chp_parameters_dict,
        on=merge_cols,
        how='left'
    )

    # Assign merged parameter values back into the main dataframe
    df.loc[mask, param_cols] = df_merge[param_cols].values

    # Save back into the dictionary
    processed_power_plants_dfs_dict[key] = df

# Print the updated columns from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPPowerToHeat,	CHPPowerLossFactor, CHPMaxHeat and COP columns for {key}:")
    print(df[['CHPPowerToHeat' , 'CHPPowerLossFactor' , 'CHPMaxHeat' , 'COP']])

CHPPowerToHeat,	CHPPowerLossFactor, CHPMaxHeat and COP columns for BE_2030:
    CHPPowerToHeat  CHPPowerLossFactor                      CHPMaxHeat  COP
0              NaN                 NaN                             NaN  NaN
1              NaN                 NaN                             NaN  NaN
2              NaN                 NaN                             NaN  NaN
3              NaN                 NaN                             NaN  NaN
4              NaN                 NaN                             NaN  NaN
5              NaN                 NaN                             NaN  NaN
6              NaN                 NaN                             NaN  NaN
7              NaN                 NaN                             NaN  NaN
8              NaN                 NaN                             NaN  NaN
9              NaN                 NaN                             NaN  NaN
10             NaN                 NaN                             NaN  NaN
11          

/tmp/ipykernel_844689/2837210924.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, param_cols] = df_merge[param_cols].values
/tmp/ipykernel_844689/2837210924.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'PowerCapacity / CHPPowerToHeat', 'Po

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Fullfilling the CHP features—CHPMaxHeat—from the CHP parameters dictionary formulas.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [80]:
# Pattern to detect words (potential variable names)
formula_pattern = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    df = processed_power_plants_dfs_dict[key]

    for idx, row in df.iterrows():

        cell = row["CHPMaxHeat"]

        # Skip empty or non-string cells
        if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == "":
            continue

        expression = cell.strip()

        # ---- 1. Extract possible variable names ----
        tokens = set(formula_pattern.findall(expression))
        columns_used = [t for t in tokens if t in df.columns]

        if not columns_used:
            # No valid column names → not a formula
            continue

        # ---- 2. Build evaluation dictionary from row ----
        local_vars = {}
        valid = True

        for col in columns_used:
            val = row[col]
            if pd.isna(val):
                valid = False
                break
            local_vars[col] = float(val)

        if not valid:
            continue

        # ---- 3. Safe evaluation ----
        try:
            # Evaluate with NO builtins and ONLY our variables
            result = eval(
                expression,
                {"__builtins__": None},   # Disable all built-ins
                local_vars                # Only allow numeric column values
            )

            df.at[idx, "CHPMaxHeat"] = float(result)

        except Exception as e:
            print(f"⚠️ Error evaluating formula in {key}, row {idx}: '{expression}' → {e}")
            continue

    processed_power_plants_dfs_dict[key] = df

# Print the updated column from the DataFrame
for key, df in processed_power_plants_dfs_dict.items():
    print(f"CHPMaxHeat column for {key}:")
    print(df[['CHPMaxHeat']])

CHPMaxHeat column for BE_2030:
    CHPMaxHeat
0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
5          NaN
6          NaN
7          NaN
8          NaN
9          NaN
10         NaN
11         NaN
12         NaN
13         NaN
14         NaN
15         NaN
16         NaN
17         NaN
18         NaN
19         NaN
20         NaN
21         NaN
22         NaN
23         NaN
24         NaN
25  611.699725
26  361.458929
27  165.668676
28  110.445784
29    0.174371
30   91.088992
31   60.725995
32  693.589866
33  462.393244
34         NaN
35         NaN
36         NaN
38         NaN
39         NaN
41         NaN
43         NaN
CHPMaxHeat column for FR_2030:
      CHPMaxHeat
0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
5            NaN
6            NaN
7            NaN
8            NaN
9            NaN
10           NaN
11           NaN
12           NaN
13           NaN
14           NaN
15           NaN
16           NaN
17

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [81]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                  {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                  {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                 {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:             {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):            {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:           tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:        overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:               fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the Processed Power Plant DataFrames (dictionary):  {list(processed_power_plants_dfs_dict.keys())}\n")
print (f"name of the dispaSET CHP technologies list:                 dispaSET_chp_tech_list\n")
print (f"Keys of the CHP type dictionary:                            {list(chp_type_dict.keys())}\n")
print (f"Keys of the CHP parameters dictionady:                      {list(chp_parameters_dict.keys())}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                  PowerPlants

Path to the Power Plants Base data folder:                  /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                 PowerPlants

Path to the Power Plants_Pypsa Raw data folder:             /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:        PowerPlants

Path to the Power Plants_Pypsa Formated data folder:        /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                          ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                                2030

Name of the Power Plant DataFrames (dictionary):            ['BE_2030', 'FR_

<div style="background-color: black;">
  <hr style="border: 1px solid skyblue;">
    
  <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman'; color: skyblue;">
    8.5. Techno-Economic Features Classification
  </div>

  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    The power units have the following features to be set:
  </div>
    
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 11px; font-family: 'Times New Roman'; color: skyblue;">
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">Efficiency</span>
    : Ratio (%) of useful energy output to energy input.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">MinEfficiency</span>
    : Minimum efficiency (%) at part-load operation.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">MinUpTime</span>
    : Minimum time (hours) a unit must remain ON once started before it can be shut down.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">MinDownTime</span>
    : Minimum time (hours) a unit must remain OFF once shut down before it can be restarted.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">RampUpRate</span>
    : Maximum rate (% of rated capacity/min) at which a unit can increase its output.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">RampDownRate</span>
    : Maximum rate (% of rated capacity/min) at which a unit can decrease its output.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">StartUpCost</span>
    : Cost (EUR) incurred when starting the unit from an OFF state.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">NoLoadCost_pu</span>
    : Cost (EUR/h) of keeping the unit ON at zero output.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">RampingCost</span>
    : Additional cost (EUR/MW) associated with changing output—ramping up or down.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">PartLoadMin</span>
    : Minimum output level (% of rated capacity) when the unit is ON.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">StartUpTime</span>
    : Time required (hours) to bring the unit from OFF to ON and ready to produce power.<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">CO2Intensity</span>
    : Amount of tons of CO₂ emitted per MWh of electricity generated.<br>
</div>

  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    To select these parameters, two stages of datasets were used.
      <br>  
    The first dataset was compiled from various sources within the available bibliographic literature:
  </div>

<div style="text-align: justify; margin-left: 2.0em; font-weight: normal; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
DIW. (2013). Current and Prospective Costs of Electricity Generation until 2030.
<br>
<a href="https://www.diw.de/documents/publikationen/73/diw_01.c.424566.de/diw_datadoc_2013-068.pdf" style="color:skyblue">https://www.diw.de/documents/publikationen/73/diw_01.c.424566.de/diw_datadoc_2013-068.pdf</a>
</li>
<li>
DIW (2014). Electricity Sector Data for Policy-Relevant Modeling.
<br>
<a href="https://www.diw.de/documents/publikationen/73/diw_01.c.440963.de/diw_datadoc_2014-072.pdf" style="color:skyblue">https://www.diw.de/documents/publikationen/73/diw_01.c.440963.de/diw_datadoc_2014-072.pdf</a>
</li>
<li>
DIW. (2016). On Start-up Costs of Thermal Power Plants in Markets with Increasing Shares of Fluctuating Renewables.
<br>
<a href="https://d-nb.info/115301274X/34" style="color:skyblue">https://d-nb.info/115301274X/34</a>
</li>
<li>
Hentschel. (2016). Fraunhofer report on power plant flexibility.
<br>
<a href="https://doi.org/10.1016/j.egyr.2016.03.002" style="color:skyblue">https://doi.org/10.1016/j.egyr.2016.03.002</a>
</li>
<li>
Bertsch. (2016). Relevance of Grid Expansion.
<br>
<a href="https://www.ewi.uni-koeln.de/cms/wp-content/uploads/2016/12/WP_15_07-Relevance-of-Grid-Expansion.pdf" style="color:skyblue">https://www.ewi.uni-koeln.de/cms/wp-content/uploads/2016/12/WP_15_07-Relevance-of-Grid-Expansion.pdf</a>
</li>
<li>
IEA. (2011). Technology Roadmap.
<br>
<a href="https://www.iea.org/reports/world-energy-outlook-2011" style="color:skyblue">https://www.iea.org/reports/world-energy-outlook-2011</a>
</li>
<li>
Klobasa et al. (2009). Dynamic simulation of load management and integration of wind energy into an electricity grid.
<br>
<a href="https://publica-rest.fraunhofer.de/server/api/core/bitstreams/8dd2c2f3-d4ab-439b-811e-35c1c99d77d0/content" style="color:skyblue">https://publica-rest.fraunhofer.de/server/api/core/bitstreams/8dd2c2f3-d4ab-439b-811e-35c1c99d77d0/content</a>
</li>
<li>
Traber, Kemfert. (2011). Gone with the wind? — Electricity market prices and incentives to invest in thermal power plants under increasing wind energy supply.
<br>
<a href="https://doi.org/10.1016/j.eneco.2010.07.002" style="color:skyblue">https://doi.org/10.1016/j.eneco.2010.07.002</a>
</li>
<li>
Van der Bergh, Delarue. (2015). Cycling of conventional power plants: Technical limits and actual costs
<br>
<a href="https://doi.org/10.1016/j.enconman.2015.03.026" style="color:skyblue">https://doi.org/10.1016/j.enconman.2015.03.026</a>
</li>
<li>
Parsons Brinckerhoff - DECC. (2014). UK DECC power plant costs report.
<br>
<a href="https://webarchive.nationalarchives.gov.uk" style="color:skyblue">https://webarchive.nationalarchives.gov.uk</a>
</li>
<li>
Ecofys. (2014). Technical Assessment of the Operation of Coal and Gas Plant.
<br>
<a href="https://assets.publishing.service.gov.uk/media/5a7dfc11ed915d74e33ef4be/Technical_Assessment_of_the_Operation_of_Coal_and_Gas_Plant_PB_Power_FIN....pdf" style="color:skyblue">https://assets.publishing.service.gov.uk/media/5a7dfc11ed915d74e33ef4be/Technical_Assessment_of_the_Operation_of_Coal_and_Gas_Plant_PB_Power_FIN....pdf</a>
</li>
<li>
dena. (2005). Deutsche Energie-Agentur reports on efficiency.
<br>
<a href="https://www.dena.de/fileadmin/dena/Dokumente/Pdf/9113_dena-Netzstudie_I.pdf" style="color:skyblue">https://www.dena.de/fileadmin/dena/Dokumente/Pdf/9113_dena-Netzstudie_I.pdf</a>
</li>
<li>
dena. (2008). Energy planning for the grid integration of onshore and offshore wind energy in Germany up to the year 2020
<br> 
<a href="https://www.boell.de/sites/default/files/assets/boell.de/images/download_de/oekologie/Kurzanalyse_KuN_Planung_D_2020_2030_Kurzfassung.pdf" style="color:skyblue">https://www.boell.de/sites/default/files/assets/boell.de/images/download_de/oekologie/Kurzanalyse_KuN_Planung_D_2020_2030_Kurzfassung.pdf</a>
</li>
<li>
RWI. (1997). The International and German Economic Situation in Fall 1997.
<br>    
<a href="https://www.iwh-halle.de/en/publications/detail/the-international-and-german-economic-situation-in-fall-1997" style="color:skyblue">https://www.iwh-halle.de/en/publications/detail/the-international-and-german-economic-situation-in-fall-1997</a>
</li>
<li>
Grimm. (2007). Electricity Market Design: On the Design of Auction Rules at the Energy Exchange (EEX).
<br>
<a href="https://link.springer.com/content/pdf/10.1007/s12398-008-0020-7.pdf" style="color:skyblue">https://link.springer.com/content/pdf/10.1007/s12398-008-0020-7.pdf</a>
</li>
<li>
Kumar et al. (2012). Median cost studies.
<br>
<a href="https://docs.nrel.gov/docs/fy12osti/55433.pdf" style="color:skyblue">https://docs.nrel.gov/docs/fy12osti/55433.pdf</a>
</li>
</ol>
</div>
 
<hr style="border: 2px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.5.1. Techno-Economic Features Dictionary 1
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The full data for the difetent units, according the technolgy and subtechnology clasification (CHP), fuel type usage and the Power capacity range is loaded.
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue">* Notes</span>
    This dictionary is currently incomplete for some technologies and is therefore undergoing continuous revision and updating
    <br>
    <span style="display: inline-block; width: 6em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue"></span>
    Additionally, the sources for some of the recently updated technologies need to be verified and added to the reference sheet in the LookUp_Table file.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [82]:
# Get the path to the "TechnoEconomic_Features_Dictionary_1" and "TechnoEconomic_Features_Dictionary_2" files.
additional_path_4 = "/scripts/Unleash_PyPSA_DispaSET_Raw_Data_Processing/Coeficients_Sources/TechnoEconomic_Features_Dictionary_1.csv"

technoeconomic_features_dictionary_1_file_path = dispaSET_unleash_folder_path + additional_path_4

print("technoeconomic_features_dictionary_1_file_path:", technoeconomic_features_dictionary_1_file_path)

# Upload the "TechnoEconomic_Features_Dictionary_1" and "TechnoEconomic_Features_Dictionary_2" dictionaries.
technoeconomic_features_dictionary_1 = pd.read_csv(technoeconomic_features_dictionary_1_file_path)

technoeconomic_features_dictionary_1_file_path: /home/ray/Dispa-SET_Unleash/scripts/Unleash_PyPSA_DispaSET_Raw_Data_Processing/Coeficients_Sources/TechnoEconomic_Features_Dictionary_1.csv


In [83]:
technoeconomic_features_dictionary_1

,Fuel,Technology,Type / Class,Age bin,PowerCapacity,Efficiency,MinUpTime,MinDownTime,RampUpRate,RampDownRate,...,RampingCost,PartLoadMin,StartUpTime,CHPType,NoLoadCost_pu,MinEfficiency,CO2Intensity,STOSelfDischarge,STOMaxChargingPower,STOChargingEfficiency
0,HRD,STUR,Sub-critical,10.0,400.0,0.397207,9.5,2.25,0.0429,0.0429,...,1.65,0.35,3.7,NaN,NaN,NaN,0.88,NaN,NaN,NaN
1,HRD,STUR,Sub-critical,10.0,400.0,0.371743,9.5,2.25,0.0429,0.0429,...,1.65,0.35,3.7,NaN,NaN,NaN,0.88,NaN,NaN,NaN
2,HRD,STUR,Sub-critical,10.0,150.0,0.371743,9.5,2.25,0.0429,0.0429,...,1.65,0.35,3.7,NaN,NaN,NaN,0.88,NaN,NaN,NaN
3,HRD,STUR,Sub-critical,10.0,150.0,0.315249,9.5,2.25,0.0429,0.0429,...,1.65,0.35,3.7,NaN,NaN,NaN,0.88,NaN,NaN,NaN
4,HRD,STUR,Sub-critical,10.0,7.5,0.315249,9.5,2.25,0.0429,0.0429,...,1.65,0.35,3.7,NaN,NaN,NaN,0.88,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,ELE,BEVS,Li-ion,NaN,200.0,0.900000,0.0,0.00,100.0000,100.0000,...,1.05,0.00,0.0,NaN,1.05,0.85,0.00,0.0015,200.0,0.97
484,ELE,BEVS,Li-ion,NaN,200.0,0.900000,0.0,0.00,100.0000,100.0000,...,1.05,0.00,0.0,NaN,1.05,0.85,0.00,0.0015,200.0,0.97
485,ELE,BEVS,Li-ion,NaN,10.0,0.900000,0.0,0.00,100.0000,100.0000,...,1.05,0.00,0.0,NaN,1.05,0.85,0.00,0.0015,10.0,0.97
486,ELE,BEVS,Li-ion,NaN,10.0,0.900000,0.0,0.00,100.0000,100.0000,...,1.05,0.00,0.0,NaN,1.05,0.85,0.00,0.0015,10.0,0.97


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering the techno-economic parameters according the dictionary 1.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [84]:
def update_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_1, zone_names, data_target_year):
    unmatched_rows = []
    numeric_cols = [
        'Efficiency'  , 'MinUpTime'   , 'MinDownTime' , 'RampUpRate'    , 'RampDownRate'  , 'StartUpCost'  , 
        'RampingCost' , 'PartLoadMin' , 'StartUpTime' , 'NoLoadCost_pu' , 'MinEfficiency' ,	'CO2Intensity'
    ]

    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue

        df = processed_power_plants_dfs_dict[key]

        for idx, row in df.iterrows():
            tech = row['Technology']
            chp = row.get('CHPType', np.nan)
            fuel = row['Fuel']
            power_capacity = row['PowerCapacity']

            # Find matching rows in the technoeconomic dictionary
            matches = technoeconomic_features_dictionary_1[
                (technoeconomic_features_dictionary_1['Technology'] == tech) &
                (technoeconomic_features_dictionary_1['CHPType'].isna() if pd.isna(chp)
                 else (technoeconomic_features_dictionary_1['CHPType'] == chp)) &
                (technoeconomic_features_dictionary_1['Fuel'] == fuel)
            ]

            if matches.empty:
                unmatched_rows.append((key, idx))
                continue

            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]

            # Calculate average for numeric columns
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()

    # Return just the number of unmatched rows
    return len(unmatched_rows)

# Example usage:
num_unmatched = update_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_1,
    zone_names,
    data_target_year
)

print("Number of unmatched rows:", num_unmatched)

Number of unmatched rows: 29


<div style="background-color: black;">
<hr style="border: 1px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.5.2. Techno-Economic Features Dictionary 2
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    The full data for the different units from 30 EU countries in the Dispa-SET database are uploaded for use in a secondary filtering process for all those rows that cannot be found data in the first dictionary
<hr style="border: 0.5px solid skyblue;">
</div>

In [85]:
# Get the path to the "TechnoEconomic_Features_Dictionary_12 file.
additional_path_5 = "/scripts/Unleash_PyPSA_DispaSET_Raw_Data_Processing/Coeficients_Sources/TechnoEconomic_Features_Dictionary_2.csv"

technoeconomic_features_dictionary_2_file_path = dispaSET_unleash_folder_path + additional_path_5

print("technoeconomic_features_dictionary_2_file_path:", technoeconomic_features_dictionary_2_file_path)

# Upload the "TechnoEconomic_Features_Dictionary_1" dictionay.
technoeconomic_features_dictionary_2 = pd.read_csv(technoeconomic_features_dictionary_2_file_path)

technoeconomic_features_dictionary_2

technoeconomic_features_dictionary_2_file_path: /home/ray/Dispa-SET_Unleash/scripts/Unleash_PyPSA_DispaSET_Raw_Data_Processing/Coeficients_Sources/TechnoEconomic_Features_Dictionary_2.csv


,PowerCapacity,Unit,Zone,Technology,Fuel,Efficiency,MinUpTime,MinDownTime,RampUpRate,RampDownRate,...,CHPPowerLossFactor,CHPMaxHeat,InitialPower,Type,JRC_id,RampUpMax,RampDownMax,RampStartUpMaximum,RampShutDownMaximum,CostFixed
0,258.000000,Aghada Unit 1,IE,STUR,GAS,0.380,4.0,4.0,0.015000,0.014961,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,444.000000,Aghada Unit 2,IE,COMC,GAS,0.530,4.0,1.0,0.019887,0.027883,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,90.000000,Aghada CT 1,IE,GTUR,GAS,0.270,4.0,1.0,0.055556,0.055556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,90.000000,Aghada CT 2,IE,GTUR,GAS,0.260,4.0,1.0,0.055556,0.055556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,90.000000,Aghada CT 4,IE,GTUR,GAS,0.270,4.0,1.0,0.055556,0.055556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
981,9.675118,HU_ICEN_GAS,HU,ICEN,GAS,0.490,0.0,0.0,1.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
982,503.265000,HU_STUR_LIG,HU,STUR,LIG,0.393,8.0,8.0,0.015210,0.017483,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983,943.395000,HU_STUR_NUC,HU,STUR,NUC,1.000,24.0,24.0,0.003333,0.003333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
984,49.000000,HU_PHOT_SUN,HU,PHOT,SUN,1.000,0.0,0.0,0.020000,0.020000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering the techno-economic parameters according the dictionary 2.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [86]:
def update_empty_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_2, zone_names, data_target_year):
    unmatched_rows = []
    numeric_cols = [
        'Efficiency'  , 'MinUpTime'   , 'MinDownTime' , 'RampUpRate'    , 'RampDownRate'  , 'StartUpCost'  ,
        'RampingCost' , 'PartLoadMin' , 'StartUpTime' , 'NoLoadCost_pu' , 'MinEfficiency' , 'CO2Intensity'
    ]

    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue

        df = processed_power_plants_dfs_dict[key]

        for idx, row in df.iterrows():
            # Check if all numeric columns are empty or NaN
            if not all(pd.isna(row[col]) or row[col] == '' for col in numeric_cols):
                continue  # Skip if any numeric column has a value

            tech = row['Technology']
            chp = row.get('CHPType', np.nan)
            fuel = row['Fuel']
            power_capacity = row['PowerCapacity']

            # Find matching rows in the technoeconomic dictionary
            matches = technoeconomic_features_dictionary_2[
                (technoeconomic_features_dictionary_2['Technology'] == tech) &
                (technoeconomic_features_dictionary_2['CHPType'].isna()
                 if pd.isna(chp)
                 else (technoeconomic_features_dictionary_2['CHPType'] == chp)) &
                (technoeconomic_features_dictionary_2['Fuel'] == fuel)
            ]

            if matches.empty:
                unmatched_rows.append((key, idx))
                continue

            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]

            # Calculate average for numeric columns
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()

    # Return only the number of unmatched rows
    return len(unmatched_rows)
    
# Example usage:
num_unmatched = update_empty_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_2,
    zone_names,
    data_target_year
)

print("Number of unmatched rows:", num_unmatched)

Number of unmatched rows: 39


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Since the initial classification used 'Fuel', 'Technology', 'PowerCapacity', and 'CHPType' as discrimination parameters to find the most accurate unit data, and because the resulting dictionaries do not cover all possible parameter combinations, the next step will involve assigning techno-economic parameters based solely on the technology type category.
<br>
These parameters will be calculated as the average of all found matches within Dictionary 1.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [87]:
def update_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_1, zone_names, data_target_year):
    unmatched_rows = []
    numeric_cols = [
        'Efficiency'  , 'MinUpTime'   , 'MinDownTime' , 'RampUpRate'    , 'RampDownRate'  , 'StartUpCost'  , 
        'RampingCost' , 'PartLoadMin' , 'StartUpTime' , 'NoLoadCost_pu' , 'MinEfficiency' ,	'CO2Intensity'
    ]
    
    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue

        df = processed_power_plants_dfs_dict[key]

        for idx, row in df.iterrows():
            tech = row['Technology']
            power_capacity = row['PowerCapacity']

            # ✅ MATCH USING ONLY TECHNOLOGY
            matches = technoeconomic_features_dictionary_1[
                technoeconomic_features_dictionary_1['Technology'] == tech
            ]

            if matches.empty:
                unmatched_rows.append((key, idx))
                continue

            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]

            # Fill numeric values with averages
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()

    return len(unmatched_rows)

# Example usage:
num_unmatched = update_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_1,
    zone_names,
    data_target_year
)

print("Number of unmatched rows:", num_unmatched)

Number of unmatched rows: 0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
This step repeats the logic from the previous cell, but uses Dictionary 2.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [88]:
def update_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_2, zone_names, data_target_year):
    unmatched_rows = []
    numeric_cols = [
        'Efficiency'  , 'MinUpTime'   , 'MinDownTime' , 'RampUpRate'    , 'RampDownRate'  , 'StartUpCost'  , 
        'RampingCost' , 'PartLoadMin' , 'StartUpTime' , 'NoLoadCost_pu' , 'MinEfficiency' ,	'CO2Intensity'
    ]

    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue

        df = processed_power_plants_dfs_dict[key]

        for idx, row in df.iterrows():
            tech = row['Technology']
            power_capacity = row['PowerCapacity']

            # ✅ MATCH USING ONLY TECHNOLOGY
            matches = technoeconomic_features_dictionary_2[
                technoeconomic_features_dictionary_2['Technology'] == tech
            ]

            if matches.empty:
                unmatched_rows.append((key, idx))
                continue

            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]

            # Fill numeric values with averages
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()

    return len(unmatched_rows)

# Example usage:
num_unmatched = update_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_2,
    zone_names,
    data_target_year
)

print("Number of unmatched rows:", num_unmatched)

Number of unmatched rows: 65


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filling the remaining empty fields for all power units.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables | Parameters | Dictionaries | Lists | Others
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [89]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the Power Plants Base data folder:                  {power_plants_base_data_folder_name}\n")
print (f"Path to the Power Plants Base data folder:                  {power_plants_base_data_folder_path}\n")
print (f"Name of Power Plants_Pypsa Raw data folder:                 {power_plants_pypsa_raw_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Raw data folder:             {power_plants_pypsa_raw_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:        {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Name of the Power Plant DataFrames (dictionary):            {list(power_plants_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:           tech_equivalences_dict\n")
print (f"Name of the Overall Fuel Technology Match DataFrame:        overal_fuel_technologies_match_df\n")
print (f"Name of the Fuel Technology Match Dictionary:               fuel_technologies_match_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the Processed Power Plant DataFrames (dictionary):  {list(processed_power_plants_dfs_dict.keys())}\n")
print (f"name of the dispaSET CHP technologies list:                 dispaSET_chp_tech_list\n")
print (f"Keys of the CHP type dictionary:                            {list(chp_type_dict.keys())}\n")
print (f"Keys of the CHP parameters dictionady:                      {list(chp_parameters_dict.keys())}\n")
print (f"Path to the Technoeconomic Features Dictionary 1 file:      {technoeconomic_features_dictionary_1_file_path}\n")
print (f"Path to the Technoeconomic Features Dictionary 2 file:      {technoeconomic_features_dictionary_2_file_path}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the Power Plants Base data folder:                  PowerPlants

Path to the Power Plants Base data folder:                  /home/ray/Dispa-SET_Unleash/Database/PowerPlants

Name of Power Plants_Pypsa Raw data folder:                 PowerPlants

Path to the Power Plants_Pypsa Raw data folder:             /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/PowerPlants

Name of the Power Plants_Pypsa Formated data folder:        PowerPlants

Path to the Power Plants_Pypsa Formated data folder:        /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants

Name of the zones:                                          ['BE', 'FR', 'DE', 'NL', 'UK']

Target year:                                                2030

Name of the Power Plant DataFrames (dictionary):            ['BE_2030', 'FR_

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
  <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman'; color: skyblue;">
    8.6. Storage Features Classification
  </div>

  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    The power units have the following storage features to be set:
  </div>
    
<div style="text-align: justify; margin-left: 3.0em; font-weight: normal; font-size: 11px; font-family: 'Times New Roman'; color: skyblue;">
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">STOCapacity</span>
    : The maximum amount of energy (or fuel) that the storage unit can hold. This defines the overall size of the reservoir ($\text{e.g., MWh}$ or $\text{GJ}$).<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">STOSelfDischarge</span>
    : The rate at which stored energy is lost over time, even when the unit is not actively charging or discharging. This represents internal leakage or standby consumption ($\text{e.g., } \%/\text{day}$ or $\text{p.u.}$).<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">STOMaxChargingPower</span>
    : The maximum rate ($\text{e.g., MW}$) at which the storage unit can absorb energy (charge). This is limited by the capacity of the charging equipment (the converter).<br>
    <span style="display: inline-block; width: 12em; font-weight: bold; font-family: 'Times New Roman'; color: skyblue;">STOChargingEfficiency</span>
    : The ratio of energy added to the storage unit to the electrical energy input required to charge it. It accounts for losses during the charging process ($\text{e.g., } 0.90$ or $90\%$).<br>
</div>
<hr style="border: 2px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.6.1.  Storage Features Power Units List
    </div>
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    To select these parameters—excluding STOCapacity at this stage—the following list identifies the power units that incorporate storage features, as defined by the Dispa-SET documentation.
</div>
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 12px; font-family: TimesNewRoman; color:skyblue">
    * Notes: &nbsp;&nbsp; This list need to be updated according to the pending or implemented changes in the<code>commons.py</code> script.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [90]:
# Define all the Dispa-SET power units whit storage features lists from the common.py script of Dispa-SET core scripts
dispaSET_storage_features_power_unit_list = [ 'HDAM' , 'HPHS' , 'BATS' , 'BEVS' , 'CAES' , 'SCSP']

# Create a DataFrame with the single column
dispaSET_storage_features_power_unit_list = pd.DataFrame(dispaSET_storage_features_power_unit_list, columns=['Dispa-SET Storage Features Power Units'])

# Print the final DataFrame
dispaSET_storage_features_power_unit_list

,Dispa-SET Storage Features Power Units
0,HDAM
1,HPHS
2,BATS
3,BEVS
4,CAES
5,SCSP


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Dictionary 1.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [91]:
def update_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_1, zone_names, data_target_year, storage_tech_list):
    unmatched_rows = []
    numeric_cols = [
        'STOSelfDischarge' , 'STOMaxChargingPower' , 'STOChargingEfficiency'
    ]
    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue
        df = processed_power_plants_dfs_dict[key]
        for idx, row in df.iterrows():
            tech = row['Technology']
            # Only process rows where Technology is in the storage_tech_list
            if tech not in storage_tech_list.values:
                continue
            power_capacity = row['PowerCapacity']
            # Match using only Technology
            matches = technoeconomic_features_dictionary_1[
                technoeconomic_features_dictionary_1['Technology'] == tech
            ]
            if matches.empty:
                unmatched_rows.append((key, idx))
                continue
            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]
            # Fill numeric values with averages
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()
    return len(unmatched_rows)

# Example usage:
num_unmatched = update_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_1,
    zone_names,
    data_target_year,
    dispaSET_storage_features_power_unit_list
)
print("Number of unmatched rows:", num_unmatched)


Number of unmatched rows: 0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Dictionary 2.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [92]:
def update_dataframes(processed_power_plants_dfs_dict, technoeconomic_features_dictionary_2, zone_names, data_target_year, storage_tech_list):
    unmatched_rows = []
    numeric_cols = [
        'STOSelfDischarge' , 'STOMaxChargingPower' , 'STOChargingEfficiency'
    ]
    for zone in zone_names:
        key = f"{zone}_{data_target_year}"
        if key not in processed_power_plants_dfs_dict:
            continue
        df = processed_power_plants_dfs_dict[key]
        for idx, row in df.iterrows():
            tech = row['Technology']
            # Only process rows where Technology is in the storage_tech_list
            if tech not in storage_tech_list.values:
                continue
            power_capacity = row['PowerCapacity']
            # Match using only Technology
            matches = technoeconomic_features_dictionary_2[
                technoeconomic_features_dictionary_2['Technology'] == tech
            ]
            if matches.empty:
                unmatched_rows.append((key, idx))
                continue
            # Find the closest PowerCapacity
            matches = matches.copy()
            matches['CapacityDiff'] = abs(matches['PowerCapacity'] - power_capacity)
            closest_matches = matches[matches['CapacityDiff'] == matches['CapacityDiff'].min()]
            # Fill numeric values with averages
            for col in numeric_cols:
                if col in closest_matches.columns:
                    df.at[idx, col] = closest_matches[col].mean()
    return len(unmatched_rows)

# Example usage:
num_unmatched = update_dataframes(
    processed_power_plants_dfs_dict,
    technoeconomic_features_dictionary_2,
    zone_names,
    data_target_year,
    dispaSET_storage_features_power_unit_list
)
print("Number of unmatched rows:", num_unmatched)

Number of unmatched rows: 5


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
When a unit's exact Power Capacity value is not available in the lookup dictionaries, an average is calculated using data from the most closely matched units. Consequently, the calculated STOMaxChargingPower must be checked to ensure it is less than or equal to the unit's corresponding Power Capacity, and the calculated STOChargingEfficiency must be checked against its respective total Efficiency.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [93]:
for zone in zone_names:
    key = f"{zone}_{data_target_year}"
    if key in processed_power_plants_dfs_dict:
        df = processed_power_plants_dfs_dict[key]
        mismatch_count = 0

        for index, row in df.iterrows():
            if row['PowerCapacity'] < row['STOMaxChargingPower'] or row['Efficiency'] < row['STOChargingEfficiency']:
                mismatch_count += 1
                if row['PowerCapacity'] < row['STOMaxChargingPower']:
                    df.at[index, 'STOMaxChargingPower'] = row['PowerCapacity']
                if row['Efficiency'] < row['STOChargingEfficiency']:
                    df.at[index, 'STOChargingEfficiency'] = row['Efficiency']

        print(f"Found {mismatch_count} rows with mismatched values for {key}")

Found 0 rows with mismatched values for BE_2030
Found 0 rows with mismatched values for FR_2030
Found 2 rows with mismatched values for DE_2030
Found 0 rows with mismatched values for NL_2030
Found 2 rows with mismatched values for UK_2030


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    8.6.2.  Power Units Storage Capacity
    </div>
  <div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman'; color: skyblue;">
    To select the 'STOCapacity' the follwing storage technology correlation will be used:
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [94]:
# Dictionary mapping Dispa-SET acronyms to PyPSA tech names

tech_storage_equivalences_dict = {
    
# -----------------------
# Renewable Power Units
# -----------------------
"generators offwind"                       :  {"tech": ["WTOF"                                                    ] ,   "fuel": ["WIN"                                                     ] ,   "storage": [" "                     ]}  ,
"generators offwind-ac"                    :  {"tech": ["WTOF"                                                    ] ,   "fuel": ["WIN"                                                     ] ,   "storage": [" "                     ]}  ,
"generators offwind-dc"                    :  {"tech": ["WTOF"                                                    ] ,   "fuel": ["WIN"                                                     ] ,   "storage": [" "                     ]}  ,
    
"generators onwind"                        :  {"tech": ["WTON"                                                    ] ,   "fuel": ["WIN"                                                     ] ,   "storage": [" "                     ]}  ,

"generators solar"                         :  {"tech": ["PHOT"                                                    ] ,   "fuel": ["SUN"                                                     ] ,   "storage": [" "                     ]}  ,
"generators solar rooftop"                 :  {"tech": ["PHOT"                                                    ] ,   "fuel": ["SUN"                                                     ] ,   "storage": [" "                     ]}  ,

"generators ror"                           :  {"tech": ["HROR"                                                    ] ,   "fuel": ["WAT"                                                     ] ,   "storage": [" "                     ]}  ,
# -----------------------
# Conventional Power Units
# -----------------------
"links Lignite"                            :  {"tech": ["STUR"                                                    ] ,   "fuel": ["LIG"                                                     ] ,   "storage": ["stores lignite"        ]}  ,
    
"links nuclear"                            :  {"tech": ["STUR"                                                    ] ,   "fuel": ["NUC"                                                     ] ,   "storage": ["stores uranium"        ]}  ,

"links CCGT"                               :  {"tech": ["COMC"                                                    ] ,   "fuel": ["GAS"  , "HYD"  , "BIO"                                   ] ,   "storage": ["stores gas"            ]}  ,
    
"links OCGT"                               :  {"tech": ["GTUR"                                                    ] ,   "fuel": ["GAS"  , "HYD"  , "OIL"  , "AMO" , "BIO" , "WST" , "OTH"  ] ,   "storage": ["stores gas"            ]}  ,

"links oil"                                :  {"tech": ["GTUR"  , "ICEN"  , "STUR"                                ] ,   "fuel": ["OIL"                                                     ] ,   "storage": ["stores oil"            ]}  ,

"links coal"                               :  {"tech": ["STUR"  , "COMC"  , "GTUR"                                ] ,   "fuel": ["HRD"  , "PEA"                                            ] ,   "storage": ["store coal"            ]}  ,
# -----------------------
# Storage Units
# -----------------------
"links V2G"                                :  {"tech": ["BEVS"                                                    ] ,   "fuel": ["ELE"                                                     ] ,   "storage": [" "                     ]}  ,
    
"links battery discharger"                 :  {"tech": ["BATS"                                                    ] ,   "fuel": ["ELE"                                                     ] ,   "storage": ["stores battery"        ]}  ,
    
"storage_units hydro"                      :  {"tech": ["HDAM"                                                    ] ,   "fuel": ["WAT"                                                     ] ,   "storage": ["storage_units hydro"   ]}  ,
    
"storage_units PHS"                        :  {"tech": ["HPHS"                                                    ] ,   "fuel": ["WAT"                                                     ] ,   "storage": ["storage_units PHS"     ]}  ,
# -----------------------
# Combined Heat and Power Units
# -----------------------
"links urban central gas CHP"              :  {"tech": ["COMC"  , "GTUR"  , "STUR"  , "ICEN"                      ] ,   "fuel": ["GAS"                                                     ] ,   "storage": ["stores gas"            ]}  ,
"links urban central gas CHP CC"           :  {"tech": ["COMC"                                                    ] ,   "fuel": ["GAS"                                                     ] ,   "storage": ["stores gas"            ]}  ,

"links urban central solid biomass CHP"    :  {"tech": ["STUR"  , "ICEN"                                          ] ,   "fuel": ["BIO"                                                     ] ,   "storage": ["stores solid biomass"  ]}  ,
"links urban central solid biomass CHP CC" :  {"tech": ["STUR"  , "ICEN"                                          ] ,   "fuel": ["BIO"                                                     ] ,   "storage": ["stores solid biomass"  ]}  ,
# -----------------------
# Sector X Units
# -----------------------
"links H2 Electrolysis"                    :  {"tech": ["PEME"  , "ALKE"  , "SOXE"                                ] ,   "fuel": ["HYD"                                                     ] ,   "storage": ["stores H2"             ]}  ,
    
"links H2 Fuel Cell"                       :  {"tech": ["PEFC"  , "SOFC"  , "MCFC"  , "PAFC"  , "ALFC"  , "REFC"  ] ,   "fuel": ["HYD"                                                     ] ,   "storage": ["stores H2"             ]}  ,
"links H2 turbine"                         :  {"tech": ["GTUR"  , "COMC"  , "ICEN"                                ] ,   "fuel": ["HYD"                                                     ] ,   "storage": ["stores H2"             ]}  ,

"links Haber-Bosch"                        :  {"tech": ["HBBS"                                                    ] ,   "fuel": ["AMO"                                                     ] ,   "storage": ["stores NH3"            ]}  ,
    
"links DAC"                                :  {"tech": ["P2GS"                                                    ] ,   "fuel": ["OTH"                                                     ] ,   "storage": ["stores co2 stored"     ]}  ,

"links methanolisation"                    :  {"tech": ["P2BS"                                                    ] ,   "fuel": ["OTH"                                                     ] ,   "storage": ["stores methanol"       ]}  ,
    
}

tech_equivalences_dict

{'generators offwind': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators offwind-ac': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators offwind-dc': {'tech': ['WTOF'], 'fuel': ['WIN']},
 'generators onwind': {'tech': ['WTON'], 'fuel': ['WIN']},
 'generators solar': {'tech': ['PHOT'], 'fuel': ['SUN']},
 'generators solar rooftop': {'tech': ['PHOT'], 'fuel': ['SUN']},
 'generators ror': {'tech': ['HROR'], 'fuel': ['WAT']},
 'links Lignite': {'tech': ['STUR'], 'fuel': ['LIG']},
 'links nuclear': {'tech': ['STUR'], 'fuel': ['NUC']},
 'links CCGT': {'tech': ['COMC'], 'fuel': ['GAS', 'HYD', 'BIO']},
 'links OCGT': {'tech': ['GTUR'],
  'fuel': ['GAS', 'HYD', 'OIL', 'AMO', 'BIO', 'WST', 'OTH']},
 'links oil': {'tech': ['GTUR', 'ICEN', 'STUR'], 'fuel': ['OIL']},
 'links coal': {'tech': ['STUR', 'COMC', 'GTUR'], 'fuel': ['HRD', 'PEA']},
 'links V2G': {'tech': ['BEVS'], 'fuel': ['ELE']},
 'links battery discharger': {'tech': ['BATS'], 'fuel': ['ELE']},
 'storage_units hydro': {'tech': ['HDAM'], '

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filling the 'STOCapacity' parameter.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [95]:
# Main logic
for zone in zone_names:
    key_pp = f"{zone}_{data_target_year}"
    key_nodal = f"{zone}_nodal_capacities_df"

    if key_pp not in processed_power_plants_dfs_dict or key_nodal not in nodal_capacities_dfs_dict:
        print(f"Skipping {key_pp} or {key_nodal} not found.")
        continue

    df_pp = processed_power_plants_dfs_dict[key_pp]
    df_nodal = nodal_capacities_dfs_dict[key_nodal]

    for index, row in df_pp.iterrows():
        tech = row['Technology']
        unit = row['Unit']

        # Check if Technology is in the Dispa-SET list
        if tech not in dispaSET_storage_features_power_unit_list['Dispa-SET Storage Features Power Units'].values:
            print(f"Row {index} in {key_pp}: Technology '{tech}' not in Dispa-SET Storage Features Power Units list.")
            continue

        # Check if Unit matches a key in tech_storage_equivalences_dict
        if unit in tech_storage_equivalences_dict:
            storage = tech_storage_equivalences_dict[unit]['storage']
            if isinstance(storage, list) and len(storage) != 1:
                print(f"Row {index} in {key_pp}: Unit '{unit}' has invalid storage entries.")
                continue
            storage = storage if isinstance(storage, list) else [storage]
        else:
            # Find the closest match with numeric suffix
            base_unit = unit.split('_')[0]
            matches = get_close_matches(base_unit, tech_storage_equivalences_dict.keys(), n=1, cutoff=0.8)
            if not matches:
                print(f"Row {index} in {key_pp}: No close match for Unit '{unit}'.")
                continue
            closest_unit = matches[0]
            storage = tech_storage_equivalences_dict[closest_unit]['storage']
            if isinstance(storage, list) and len(storage) != 1:
                print(f"Row {index} in {key_pp}: Closest Unit '{closest_unit}' has invalid storage entries.")
                continue
            storage = storage if isinstance(storage, list) else [storage]

        # Get the capacity from nodal_capacities_dfs_dict
        tech_match = df_nodal[df_nodal['tech'] == storage[0]]
        if tech_match.empty:
            print(f"Row {index} in {key_pp}: No matching tech '{storage[0]}' in nodal capacities.")
            continue

        capacity = tech_match[str(data_target_year)].values[0]
        df_pp.at[index, 'STOCapacity'] = capacity

# Output the updated DataFrames
for key, df in processed_power_plants_dfs_dict.items():
    print(f"\nUpdated DataFrame for {key}:")
    print(df)

Row 0 in BE_2030: Technology 'WTOF' not in Dispa-SET Storage Features Power Units list.
Row 1 in BE_2030: Technology 'WTOF' not in Dispa-SET Storage Features Power Units list.
Row 2 in BE_2030: Technology 'WTOF' not in Dispa-SET Storage Features Power Units list.
Row 3 in BE_2030: Technology 'WTON' not in Dispa-SET Storage Features Power Units list.
Row 4 in BE_2030: Technology 'HROR' not in Dispa-SET Storage Features Power Units list.
Row 5 in BE_2030: Technology 'PHOT' not in Dispa-SET Storage Features Power Units list.
Row 6 in BE_2030: Technology 'PHOT' not in Dispa-SET Storage Features Power Units list.
Row 7 in BE_2030: Technology 'P2GS' not in Dispa-SET Storage Features Power Units list.
Row 8 in BE_2030: Technology 'PEME' not in Dispa-SET Storage Features Power Units list.
Row 9 in BE_2030: Technology 'ALKE' not in Dispa-SET Storage Features Power Units list.
Row 10 in BE_2030: Technology 'SOXE' not in Dispa-SET Storage Features Power Units list.
Row 11 in BE_2030: Technology '

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
For the specific case of Electric Vehicle (EV) technology, since no data was found in the PyPSA database, the STOCapacity value will be estimated using the following correlation:
\[
\text{STOCapacity} = P_{\text{EV}} \times \text{E/P}
\]
\[
\text{E/P} = 7 \ \text{h}
\]
where $P_{\text{EV}}$ is the aggregated charging/discharging power of the EV fleet (MW)
<br>
STOCapacity is the aggregated usable storage capacity of the EV fleet (MWh)
<br>
and E/P is the effective energy-to-power ratio of the EV fleet (h), here assumed to be 7 h.
<ol style="margin-top: 0; padding-left: 3.5em;">
<li>
Fraunhofer ISE & ISI. (2024). Potential of a full EV-power-system-integration in Europe and how to realise it.
<br>
<a href="https://www.transportenvironment.org/uploads/files/2024_10_Study_V2G_EU-Potential_Final.pdf" style="color:skyblue">https://www.transportenvironment.org/uploads/files/2024_10_Study_V2G_EU-Potential_Final.pdf</a>
</li>
</ol>
</div>
<hr style="border: 0.5px solid skyblue;">
</div>


In [96]:
for zone in zone_names:
    # Construct the key for the dictionary
    key = f"{zone}_{data_target_year}"

    # Check if the key exists in the dictionary
    if key in processed_power_plants_dfs_dict:
        df = processed_power_plants_dfs_dict[key]

        # Identify rows where 'Technology' is 'BEVS'
        bevs_mask = df['Technology'] == 'BEVS'

        # Multiply 'PowerCapacity' by 7 and update 'STOCapacity'
        df.loc[bevs_mask, 'STOCapacity'] = df.loc[bevs_mask, 'PowerCapacity'] * 7

        # Optionally, update the DataFrame in the dictionary
        processed_power_plants_dfs_dict[key] = df

processed_power_plants_dfs_dict

{'BE_2030':     Unnamed: 0                                        Unit  PowerCapacity  \
 0          NaN                          generators offwind    2261.800000   
 1          NaN                       generators offwind-ac    1738.199374   
 2          NaN                       generators offwind-dc    1999.999446   
 3          NaN                           generators onwind    5999.998849   
 4          NaN                              generators ror      59.015340   
 5          NaN                            generators solar   11999.999412   
 6          NaN                    generators solar rooftop    1999.996368   
 7          NaN                                   links DAC       0.007254   
 8          NaN                     links H2 Electrolysis_1      85.714286   
 9          NaN                     links H2 Electrolysis_2      42.857143   
 10         NaN                     links H2 Electrolysis_3      21.428571   
 11         NaN                        links H2 Fuel 

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
9. Final Formatted Data Storage
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Once the cleaned data has been obtained, it must be stored in the appropriate location within the Dispa-SET database file structure.
    <br>
A dedicated folder, named "Database_PyPSA", has been created to house all data related to the NEGAWAT project.
    <br>
This folder contains two primary subfolders corresponding to the project scenarios: "Reference_Scenario" and "Suficiency_Scenario".
        <br>
Each scenario subfolder contains nested sub-subfolders, each named using the acronym of a corresponding EU country.
        <br>
The processed and cleaned DataFrames are stored within their respective country sub-subfolders. Each DataFrame is converted into a .csv file and named after the processed year (e.g., `2030.csv`).
    </div>
    <hr style="border: 0.5px solid skyblue;">
</div>

In [97]:
# Ensure the base directory exists
os.makedirs(power_plants_pypsa_formated_data_folder_path, exist_ok=True)

# Save each DataFrame to its corresponding subfolder
for key, df in processed_power_plants_dfs_dict.items():
    # Extract the zone from the key (e.g., 'BE_2030' -> 'BE')
    zone = key.split('_')[0]

    # Check if the zone is in the zone_names list
    if zone not in zone_names:
        print(f"Zone '{zone}' not in zone_names list. Skipping.")
        continue

    # Create the subfolder path
    subfolder_path = os.path.join(power_plants_pypsa_formated_data_folder_path, zone)
    os.makedirs(subfolder_path, exist_ok=True)

    # Define the filename
    filename = f"{data_target_year}.csv"
    filepath = os.path.join(subfolder_path, filename)

    # Save the DataFrame as a CSV file
    df.to_csv(filepath, index=False)
    print(f"Saved {key} to {filepath}")

Saved BE_2030 to /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants/BE/2030.csv
Saved FR_2030 to /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants/FR/2030.csv
Saved DE_2030 to /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants/DE/2030.csv
Saved NL_2030 to /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants/NL/2030.csv
Saved UK_2030 to /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/PowerPlants/UK/2030.csv


<hr style="border: 4px solid skyblue;">